# Tahap 2 sampai 5: Eksplorasi, Praproses, Validasi, *Baseline*, dan Model Lanjutan
**Proyek:** Peramalan PM2.5 Beijing (Muhammad Alif Qadri, D082261018)

Tujuan bagian ini sederhana: **mengenal data, menyiapkannya, menetapkan acuan (*baseline*), lalu membandingkan model lanjutan dengan acuan itu** lewat *cross-validation* pada data latih. Data uji tidak dibuka di *notebook* ini.

**Istilah yang dipakai** (supaya tidak tertukar):

| Istilah | Arti |
|---|---|
| Stasiun latih | 9 stasiun yang datanya dipakai untuk melatih model |
| Stasiun uji | 3 stasiun yang **belum pernah dilihat model** saat dilatih (Dingling, Wanliu, Shunyi) |
| Periode latih | 80% waktu awal (sampai 2016-05-12 18:00), sama untuk semua stasiun |
| Periode uji | 20% waktu akhir (mulai 2016-05-12 19:00), sama untuk semua stasiun |
| Data latih | stasiun latih pada periode latih |
| Data uji | 12 stasiun pada periode uji, dilaporkan terpisah untuk stasiun latih dan stasiun uji |
| Tidak dipakai | stasiun uji pada periode latih (hanya dilihat untuk memilih stasiun uji dan menggambarkan pergeseran) |

Aturan yang dipegang sepanjang *notebook*:
1. Data uji disisihkan **di awal** dan tidak dilihat.
2. Model memprediksi PM2.5 **24 jam ke depan**, jadi fitur hanya boleh memakai data sampai jam sekarang.
3. Tidak ada langkah yang memakai nilai dari masa depan (misalnya interpolasi).
4. Validasi berurutan waktu memakai *gap* 24 jam antara data latih dan data validasi. *Fold* dibuat dari **jam unik**, bukan dari baris gabungan stasiun.

Nomor bagian mengikuti struktur *notebook* dari dosen:

| No | Isi |
|---|---|
| 00 | Pengaturan |
| 01 | Rumusan masalah (ringkas) |
| 02 | Muat data, sisihkan data uji dan stasiun uji, lalu eksplorasi (EDA) |
| 03 | Pembagian data dan skema validasi |
| 04 | Fitur, *pipeline* praproses, *baseline*, dan pemeriksaan kesesuaian desain |
| 05 | Model lanjutan dan *tuning* (hasil *cross-validation*, data uji tidak dibuka), termasuk eksperimen lanjutan fitur, model, dan transformasi target (05.5) |

Status setiap keputusan:
* **[Panduan]** dari dosen: data berurutan waktu tidak diacak, `TimeSeriesSplit`, *naive forecast* sebagai *baseline*, data uji dibuka satu kali.
* **[Keputusan penulis]** bukan aturan dosen: horizon 24 jam, nilai ekstrem dipertahankan (14 Nov 2026), tanpa *forward fill* (15 Nov 2026, dasarnya dihitung ulang di 04.6), 9 stasiun latih dan 3 stasiun uji dengan satu batas waktu untuk semua stasiun, tiga stasiun uji = Dingling, Wanliu, Shunyi (ditetapkan, dasarnya aturan di 02.2), dan identitas stasiun tidak dipakai sebagai fitur (16 Nov 2026).
* **Belum dikonfirmasi dosen:** desain 9 stasiun latih dan 3 stasiun uji.

Catatan urutan: data uji disisihkan **di awal bagian 02**, sebelum eksplorasi, sesuai Kode 1 panduan
("data uji disisihkan sekarang, sebelum eksplorasi lebih jauh"). Bagian 03 merinci aturan pembagiannya.

## 00. Pengaturan

In [ ]:
# Sumber data:
# 1) Google Drive pemilik (folder "Dataset PM25 Beijing"), jika ada.
# 2) Folder Data/Data Mentah di repo ini (dijalankan dari akar repo atau dari folder Notebook).
# 3) Jika keduanya tidak ada (misalnya dijalankan dosen), repo GitHub publik diunduh.
import os, subprocess
REPO = "https://github.com/LippyyDev/Tugas-Machine-Learning-PM25-Beijing"

FOLDER = os.environ.get("FOLDER_DATA")
if FOLDER is None:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        calon = "/content/drive/MyDrive/Dataset PM25 Beijing"
        FOLDER = calon if os.path.isdir(calon) else None
    except Exception:
        pass
if FOLDER is None:
    for calon in ("Data/Data Mentah", "../Data/Data Mentah"):
        if os.path.isdir(calon):
            FOLDER = calon                                # dijalankan dari dalam repo
            break
    else:
        if not os.path.isdir("repo_pm25"):
            subprocess.run(["git", "clone", "--depth", "1", REPO, "repo_pm25"], check=True)
        FOLDER = "repo_pm25/Data/Data Mentah"

# Keluaran meniru struktur repo: Hasil/Gambar, Hasil/Tabel, Data/Data Olahan.
# Di repo, akar = folder yang memuat "Data". Di Drive, akar = folder data itu sendiri.
if os.path.basename(os.path.normpath(FOLDER)) == "Data Mentah":
    AKAR = os.path.abspath(os.path.join(FOLDER, "..", ".."))
else:
    AKAR = os.path.abspath(FOLDER)
GAMBAR = os.path.join(AKAR, "Hasil", "Gambar")
TABEL = os.path.join(AKAR, "Hasil", "Tabel")
OLAHAN = os.path.join(AKAR, "Data", "Data Olahan")
for d in (GAMBAR, TABEL, OLAHAN):
    os.makedirs(d, exist_ok=True)
print("Folder data   :", FOLDER)
print("Keluaran disimpan di Hasil/Gambar, Hasil/Tabel, dan Data/Data Olahan di bawah folder:",
      os.path.basename(AKAR))

In [ ]:
import glob, sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle, Patch
import sklearn

SEED = 42
np.random.seed(SEED)
HORIZON = 24                     # jam ke depan yang diprediksi
KOLOM_ANGKA = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "WSPM"]

# Warna: biru = data utama, oranye = sorotan, abu-abu = pembanding atau tidak dipakai
BIRU, ORANYE, ABU = "#2a78d6", "#eb6834", "#b4b2a9"
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 10, "figure.dpi": 100})

# Nama kolom yang mudah dibaca untuk grafik
NAMA_KOLOM = {"PM2.5": "PM2.5 (partikel halus)", "PM10": "PM10 (partikel kasar)", "SO2": "SO2", "NO2": "NO2", "CO": "CO",
              "O3": "O3", "TEMP": "TEMP (suhu)", "PRES": "PRES (tekanan udara)", "DEWP": "DEWP (titik embun)",
              "RAIN": "RAIN (hujan)", "WSPM": "WSPM (kecepatan angin)", "wd": "wd (arah angin)"}

def judul(ax, utama, keterangan=None):
    # judul tebal di kiri, satu kalimat cara membaca di bawahnya
    ax.set_title(utama, loc="left", fontsize=12, fontweight="bold", pad=28 if keterangan else 10)
    if keterangan:
        ax.text(0, 1.025, keterangan, transform=ax.transAxes, fontsize=9, color="#52514e", va="bottom")

def angka(x):
    return f"{x:,.0f}".replace(",", ".")            # 35064 menjadi 35.064

def simpan_gambar(nama):
    plt.tight_layout()
    plt.savefig(os.path.join(GAMBAR, nama), dpi=150)
    plt.show()

print("Python", sys.version.split()[0], "| pandas", pd.__version__,
      "| numpy", np.__version__, "| scikit-learn", sklearn.__version__,
      "| matplotlib", matplotlib.__version__)

## 01. Rumusan masalah (ringkas)
* **Target:** konsentrasi PM2.5 pada **24 jam ke depan** di setiap stasiun. Satu model dilatih dari gabungan **9 stasiun latih**.
* **Fitur:** hanya data yang sudah tersedia saat prediksi dibuat (polutan, cuaca, waktu), dihitung **per stasiun**, tanpa mencampur deret antar stasiun.
* **Pengujian:** pada 20% periode terakhir, dilaporkan terpisah untuk 9 stasiun latih dan 3 stasiun uji (stasiun yang tidak pernah dilihat model).
* **Metrik utama:** RMSE, karena galat besar saat polusi tinggi paling merugikan. MAE sebagai pendamping.

Rumusan lengkap ada di proposal.

## 02. Muat data dan eksplorasi
### 02.1 Muat data
Ada 12 berkas CSV, satu per stasiun. Semuanya digabung menjadi satu tabel.

In [ ]:
berkas = sorted(glob.glob(os.path.join(FOLDER, "*.csv")))
df = pd.concat([pd.read_csv(f) for f in berkas], ignore_index=True)
df["waktu"] = pd.to_datetime(df[["year", "month", "day", "hour"]])
df = df.sort_values(["station", "waktu"]).reset_index(drop=True)

per_stasiun = df.groupby("station")["waktu"].agg(baris="size", jam_unik="nunique")
assert (per_stasiun["baris"] == per_stasiun["jam_unik"]).all()      # tidak ada jam ganda dalam satu stasiun
print(f"{len(berkas)} berkas, {df.shape[0]:,} baris, {df.shape[1]} kolom")
print(f"Baris per stasiun: {per_stasiun['baris'].min():,} sampai {per_stasiun['baris'].max():,}")
print("Rentang waktu:", df["waktu"].min(), "sampai", df["waktu"].max())
df.head(3)

In [ ]:
# Grafik ini hanya memakai jumlah baris dan waktu. Nilai PM2.5 belum disentuh.
urut = per_stasiun["baris"].sort_values()
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.barh(urut.index, urut, color=BIRU)
for y, v in enumerate(urut):
    ax.text(v + urut.max() * 0.01, y, angka(v), va="center", fontsize=9)
sama = "Jumlahnya sama di semua stasiun." if urut.nunique() == 1 else "Jumlahnya tidak sama antar stasiun."
judul(ax, "Data yang dimuat: jumlah jam tercatat per stasiun",
      f"{len(urut)} stasiun, data per jam dari {df['waktu'].min():%Y-%m-%d} sampai {df['waktu'].max():%Y-%m-%d}. {sama}")
ax.set(xlabel="Jumlah jam (satu baris = satu jam)", xlim=(0, urut.max() * 1.12))
ax.xaxis.set_major_formatter(lambda x, _: angka(x))
ax.grid(axis="y", visible=False)
simpan_gambar("t2_01_jam_per_stasiun.png")

### 02.2 Sisihkan data uji dan stasiun uji (sebelum eksplorasi)
Dua pemotongan dibuat di sini, sebelum eksplorasi:

1. **Menurut waktu:** data uji = **20% periode terakhir**, tanpa diacak. Batas waktunya **sama untuk semua stasiun** (satu stempel waktu), supaya setiap stasiun diuji pada periode yang sama dan tidak ada jam uji yang ikut terlatih lewat stasiun lain. Stasiun Beijing saling berkorelasi tinggi pada jam yang sama, jadi memotong per stasiun akan membocorkan masa depan.
2. **Menurut stasiun:** 3 dari 12 stasiun dijadikan **stasiun uji**. Model tidak pernah dilatih atau divalidasi dengan baris dari stasiun itu. Hanya 20% periode terakhirnya yang dipakai, sebagai uji "stasiun yang belum pernah dilihat" (pergeseran distribusi, tantangan wajib ketiga).

| Blok | Stasiun | Periode | Dipakai untuk |
|---|---|---|---|
| Latih | 9 stasiun latih | 80% awal | melatih dan validasi berurutan waktu |
| Uji pada stasiun latih | 9 stasiun latih | 20% akhir | uji akhir, stasiun yang dikenal model |
| Tidak dipakai | 3 stasiun uji | 80% awal | tidak melatih dan tidak memvalidasi |
| Uji pada stasiun uji | 3 stasiun uji | 20% akhir | uji akhir, stasiun yang belum pernah dilihat model |

**Tiga stasiun uji: Dingling, Wanliu, Shunyi [Keputusan penulis, 16 Nov 2026, ditetapkan sebelum data uji dibuka].** Dasarnya aturan berikut:
* Stasiun diurutkan menurut jarak distribusi PM2.5 (jarak Wasserstein) terhadap gabungan 11 stasiun lain, hanya dari periode **sebelum** batas waktu.
* Yang dijadikan stasiun uji: urutan ke-1 (paling berbeda), ke-6 (sedang), dan ke-12 (paling mirip), supaya uji mencakup beberapa tingkat pergeseran.
* Tidak ada model yang dilatih untuk memilih, dan data uji tidak dilihat.

**Batasnya:** pemilihan memakai data periode latih dari stasiun yang kemudian dijadikan stasiun uji, jadi hasil uji pada tiga stasiun itu adalah tiga contoh, bukan perkiraan umum untuk "stasiun mana pun".

Sel di bawah menghitung ulang aturan itu dan memastikan hasilnya sama dengan daftar yang ditetapkan. Mengganti daftar berarti mengubah keputusan, dan seluruh *notebook* harus dijalankan ulang dari awal.

In [ ]:
jam_unik = np.sort(df["waktu"].unique())
assert (np.diff(jam_unik) == np.timedelta64(1, "h")).all()    # jam berurutan tanpa lompatan, jadi 24 baris unik = 24 jam
BATAS = pd.Timestamp(jam_unik[int(len(jam_unik) * 0.8)])      # jam pertama periode uji, sama untuk semua stasiun

pra = df[df["waktu"] < BATAS]      # semua 12 stasiun sebelum BATAS: HANYA untuk memilih stasiun uji dan menggambarkan pergeseran
print("Batas data uji (jam pertama periode uji):", BATAS)

In [ ]:
from scipy.stats import ks_2samp, wasserstein_distance

# Langkah 1: jarak distribusi PM2.5 tiap stasiun terhadap 11 stasiun lain (hanya periode sebelum BATAS)
tab_dist = []
for s, g in pra.groupby("station"):
    v = g["PM2.5"].dropna().values
    lain = pra.loc[pra["station"] != s, "PM2.5"].dropna().values
    tab_dist.append(dict(stasiun=s, rata_rata=v.mean(), median=np.median(v), sd=v.std(), kuantil90=np.quantile(v, .9),
                         hilang_pct=100 * g["PM2.5"].isna().mean(),
                         wasserstein=wasserstein_distance(v, lain), KS=ks_2samp(v, lain).statistic))
tab_dist = pd.DataFrame(tab_dist).set_index("stasiun")
piv = pra.pivot_table(index="waktu", columns="station", values="PM2.5")
tab_dist["korelasi_dg_lainnya"] = [piv[s].corr(piv.drop(columns=s).mean(axis=1)) for s in tab_dist.index]
tab_dist = tab_dist.sort_values("wasserstein", ascending=False)
tab_dist.round(3)

In [ ]:
# Langkah 2: daftar stasiun uji yang DITETAPKAN, lalu dicek terhadap aturan
STASIUN_UJI = ["Dingling", "Wanliu", "Shunyi"]    # DITETAPKAN: keputusan penulis 16 Nov 2026
URUTAN_UJI = (1, 6, 12)                           # dasar aturan: urutan jarak Wasserstein dari terbesar (1) ke terkecil (12)

dari_aturan = [tab_dist.index[i - 1] for i in URUTAN_UJI]
assert sorted(dari_aturan) == sorted(STASIUN_UJI), (
    f"Aturan urutan {URUTAN_UJI} menghasilkan {dari_aturan}, berbeda dari daftar yang ditetapkan {STASIUN_UJI}")
STASIUN_LATIH = [s for s in sorted(df["station"].unique()) if s not in STASIUN_UJI]
assert len(STASIUN_UJI) == 3 and set(STASIUN_UJI) <= set(tab_dist.index)
assert len(STASIUN_LATIH) == 9 and not set(STASIUN_LATIH) & set(STASIUN_UJI)

tab_dist["kelompok"] = np.where(tab_dist.index.isin(STASIUN_UJI), "uji", "latih")
tab_dist = tab_dist.round(3)
tab_dist.to_csv(os.path.join(TABEL, "t2_tabel_distribusi_stasiun.csv"))
# Grafik: seberapa beda sebaran PM2.5 tiap stasiun (hanya periode sebelum BATAS)
fig, ax = plt.subplots(figsize=(9, 4.6))
nama_urut = [f"{i}. {s}" for i, s in enumerate(tab_dist.index, 1)]
warna = [ORANYE if k == "uji" else BIRU for k in tab_dist["kelompok"]]
ax.barh(nama_urut, tab_dist["wasserstein"], color=warna)
for y, v in enumerate(tab_dist["wasserstein"]):
    ax.text(v + tab_dist["wasserstein"].max() * 0.01, y, f"{v:.1f}", va="center", fontsize=9)
ax.invert_yaxis()
judul(ax, "Stasiun uji: seberapa beda sebaran PM2.5 tiap stasiun dari 11 stasiun lain",
      "Makin panjang batang, makin berbeda. Oranye = stasiun uji, diambil dari urutan ke-1, ke-6, dan ke-12.")
ax.set(xlabel="Jarak Wasserstein PM2.5 (µg/m³), hanya data sebelum batas data uji",
       xlim=(0, tab_dist["wasserstein"].max() * 1.12))
ax.legend(handles=[Patch(color=ORANYE, label="Stasiun uji (tidak dipakai melatih)"), Patch(color=BIRU, label="Stasiun latih")],
          loc="lower right", frameon=False)
ax.grid(axis="y", visible=False)
simpan_gambar("t2_03_jarak_stasiun_uji.png")
kor_pasangan = np.median(piv.corr().values[np.triu_indices(piv.shape[1], k=1)])
print("Stasiun uji (ditetapkan, sama dengan hasil aturan):", STASIUN_UJI)
print("Stasiun latih  :", STASIUN_LATIH)
print(f"Median korelasi PM2.5 antar pasangan stasiun pada jam yang sama (sebelum BATAS): {kor_pasangan:.2f}")
tab_dist

In [ ]:
latih = df[df["station"].isin(STASIUN_LATIH) & (df["waktu"] < BATAS)].copy()
blok = {
    "Latih (9 stasiun latih, 80% awal)":           len(latih),
    "Uji pada stasiun latih (9 stasiun, 20% akhir)":    (df["station"].isin(STASIUN_LATIH) & (df["waktu"] >= BATAS)).sum(),
    "Tidak dipakai (3 stasiun uji, 80% awal)":   (df["station"].isin(STASIUN_UJI) & (df["waktu"] < BATAS)).sum(),
    "Uji pada stasiun uji (3 stasiun, 20% akhir)":    (df["station"].isin(STASIUN_UJI) & (df["waktu"] >= BATAS)).sum(),
}
assert sum(blok.values()) == len(df)
for nama, n in blok.items():
    print(f"{nama:<46}{n:>9,} baris  {n / len(df):>5.0%}")
print("\nMulai di sini, eksplorasi hanya memakai `latih` (9 stasiun latih, sebelum BATAS).")

In [ ]:
# Diagram pembagian data. Hanya memakai waktu dan jumlah baris, nilai PM2.5 data uji tidak disentuh.
n_blok = list(blok.values())
tinggi_latih = len(STASIUN_LATIH) / 12
lebar_awal = 0.8
sel = [  # (kiri, bawah, lebar, tinggi, peran, jumlah, warna, warna teks)
    (0.0, 1 - tinggi_latih, lebar_awal, tinggi_latih, "LATIH", n_blok[0], BIRU, "white"),
    (lebar_awal, 1 - tinggi_latih, 1 - lebar_awal, tinggi_latih, "UJI", n_blok[1], ORANYE, "#0b0b0b"),
    (0.0, 0.0, lebar_awal, 1 - tinggi_latih, "TIDAK DIPAKAI", n_blok[2], "#d9d7d0", "#0b0b0b"),
    (lebar_awal, 0.0, 1 - lebar_awal, 1 - tinggi_latih, "UJI", n_blok[3], ORANYE, "#0b0b0b"),
]
fig, ax = plt.subplots(figsize=(9, 4.6))
for kiri, bawah, lebar, tinggi, peran, n, warna, teks in sel:
    ax.add_patch(Rectangle((kiri + 0.004, bawah + 0.006), lebar - 0.008, tinggi - 0.012, color=warna))
    ax.text(kiri + lebar / 2, bawah + tinggi / 2, f"{peran}\n{angka(n)} baris\n({n / len(df):.0%})",
            ha="center", va="center", fontsize=10, color=teks, fontweight="bold" if peran != "TIDAK DIPAKAI" else "normal")
ax.text(-0.01, 1 - tinggi_latih / 2, f"{len(STASIUN_LATIH)} stasiun latih", ha="right", va="center", fontsize=10)
ax.text(-0.01, (1 - tinggi_latih) / 2, f"{len(STASIUN_UJI)} stasiun uji", ha="right", va="center", fontsize=10)
ax.text(lebar_awal / 2, 1.02, f"80% waktu awal (sebelum {BATAS:%Y-%m-%d %H:%M})", ha="center", fontsize=10)
ax.text(lebar_awal + (1 - lebar_awal) / 2, 1.02, "20% waktu akhir", ha="center", fontsize=10)
ax.set(xlim=(-0.2, 1), ylim=(0, 1.08))
ax.axis("off")
ax.set_title("Pembagian data: latih, uji, dan stasiun uji", loc="left", fontsize=12, fontweight="bold", pad=28)
ax.text(-0.2, 1.1, "Luas kotak sebanding dengan jumlah baris. Hanya waktu dan jumlah baris yang dipakai, nilai PM2.5 data uji tidak dibuka.",
        transform=ax.transData, fontsize=9, color="#52514e", va="bottom")
simpan_gambar("t2_02_pembagian_data.png")

### 02.3 Statistik deskriptif (9 stasiun latih, digabung)

In [ ]:
latih[["PM2.5", "PM10", "TEMP", "DEWP", "PRES", "WSPM", "RAIN"]].describe().round(1)

### 02.4 Sebaran PM2.5
Seberapa sering PM2.5 bernilai rendah, dan seberapa sering sangat tinggi? Ini dasar keputusan soal nilai ekstrem.

In [ ]:
pm = latih["PM2.5"].dropna()
BATAS_TAMPIL = 500                       # nilai di atas ini tidak digambar supaya bentuk utamanya terlihat
fig, ax = plt.subplots()
ax.hist(pm[pm <= BATAS_TAMPIL], bins=50, color=BIRU, edgecolor="white", linewidth=0.5)
ax.axvline(pm.median(), color=ORANYE, lw=2, label=f"Median = {pm.median():.0f} (separuh jam di bawah nilai ini)")
ax.axvline(pm.mean(), color="#0b0b0b", lw=1.5, ls="--", label=f"Rata-rata = {pm.mean():.0f}")
ax.legend(loc="upper right", frameon=False)
ax.text(0.98, 0.62, f"{(pm > BATAS_TAMPIL).sum()} jam bernilai di atas {BATAS_TAMPIL}\nTidak digambar, tetapi tetap dipakai",
        transform=ax.transAxes, ha="right", fontsize=9, color="#52514e")
judul(ax, "Sebaran PM2.5 per jam (data latih)",
      "Rata-rata lebih tinggi dari median karena ada jam yang bernilai sangat tinggi.")
ax.set(xlabel="Konsentrasi PM2.5 (µg/m³)", ylabel="Jumlah jam (9 stasiun digabung)")
simpan_gambar("t2_04_sebaran_pm25.png")
print(f"Rata-rata {pm.mean():.1f} | median {pm.median():.1f} | kemiringan (skewness) {pm.skew():.2f}")
print(f"Jam bernilai di atas 500: {(pm > 500).sum()} dari {len(pm):,}")
pm.describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(1).to_frame("PM2.5")

### 02.5 Nilai ekstrem
Nilai sangat tinggi bisa berarti episode polusi nyata atau galat sensor.
Kita cek di 9 stasiun latih (data latih), lalu lihat apakah nilai itu muncul sendirian atau bersamaan di banyak stasiun.

In [ ]:
pm_semua = latih["PM2.5"]
print("Jumlah jam-stasiun dengan PM2.5:")
for ambang in [500, 700, 900]:
    print(f"  di atas {ambang}: {(pm_semua > ambang).sum():>4}")
print("  tepat 999   :", (pm_semua == 999).sum())

# Apakah nilai > 700 terjadi serempak? Hitung berapa stasiun yang juga > 300 pada jam itu.
jam_ekstrem = latih.loc[latih["PM2.5"] > 700, "waktu"].unique()
serempak = (latih[latih["waktu"].isin(jam_ekstrem)]
            .groupby("waktu")["PM2.5"].apply(lambda s: (s > 300).sum()))
print(f"\nPada {len(jam_ekstrem)} jam dengan PM2.5 > 700 di suatu stasiun, "
      f"median jumlah stasiun yang juga > 300: {serempak.median():.0f} dari {latih['station'].nunique()}")

### 02.6 Pola harian dan musiman
Kapan polusi paling tinggi? Pola ini menjadi dasar fitur kalender (jam dan bulan).

In [ ]:
per_jam = latih.groupby(latih["waktu"].dt.hour)["PM2.5"].mean()
nama_bulan = ["Jan","Feb","Mar","Apr","Mei","Jun","Jul","Agu","Sep","Okt","Nov","Des"]
per_bulan = latih.groupby(latih["waktu"].dt.month)["PM2.5"].mean()

print(f"Jam tertinggi: {per_jam.idxmax()}:00 ({per_jam.max():.0f}) | "
      f"terendah: {per_jam.idxmin()}:00 ({per_jam.min():.0f})")
print(f"Bulan tertinggi: {nama_bulan[per_bulan.idxmax()-1]} ({per_bulan.max():.0f}) | "
      f"terendah: {nama_bulan[per_bulan.idxmin()-1]} ({per_bulan.min():.0f})")

### 02.7 Nilai hilang
Dua hal yang dicek: **berapa banyak** yang hilang, dan **berapa lama** rentang nilai hilang berturut-turut (dihitung per stasiun, lalu digabung).
Lama rentang nilai hilang menentukan seberapa berbahaya tebakan pengisiannya (dibahas di 04.1).

In [ ]:
hilang = (latih.drop(columns=["No", "year", "month", "day", "hour", "station", "waktu"])
            .isna().mean().mul(100).sort_values())
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.barh([NAMA_KOLOM.get(k, k) for k in hilang.index], hilang,
        color=[ORANYE if k == "PM2.5" else BIRU for k in hilang.index])
for y, v in enumerate(hilang):
    ax.text(v + 0.05, y, f"{v:.1f}%", va="center", fontsize=9)
judul(ax, "Persentase jam yang datanya kosong, per kolom (data latih)",
      "Makin panjang batang, makin sering sensor tidak mencatat. Oranye = PM2.5, yang ingin diprediksi.")
ax.set(xlabel="Persen jam yang kosong (%)", xlim=(0, hilang.max() * 1.15))
ax.grid(axis="y", visible=False)
simpan_gambar("t2_05_nilai_hilang_per_kolom.png")
tabel_hilang = hilang.round(2).to_frame("% hilang")
tabel_hilang.to_csv(os.path.join(TABEL, "t2_tabel_nilai_hilang.csv"))
tabel_hilang

In [ ]:
# Lama setiap rentang nilai hilang berturut-turut pada PM2.5, dihitung di dalam tiap stasiun
def lama_nilai_hilang(seri):
    kosong = seri.isna()
    kelompok = (kosong != kosong.shift()).cumsum()
    return kosong.groupby(kelompok).sum()[lambda s: s > 0]

rentang_hilang = pd.concat([lama_nilai_hilang(g.sort_values("waktu")["PM2.5"]) for _, g in latih.groupby("station")],
                  ignore_index=True)
kelas = pd.cut(rentang_hilang, [0, 1, 3, 6, 24, 10_000],
               labels=["1 jam", "2-3 jam", "4-6 jam", "7-24 jam", "lebih dari 24 jam"])
hitung = kelas.value_counts().sort_index()

fig, ax = plt.subplots()
ax.bar(hitung.index.astype(str), hitung, color=BIRU, width=0.6)
for x, y in zip(range(len(hitung)), hitung):
    ax.text(x, y, str(y), ha="center", va="bottom")
judul(ax, "Berapa lama data PM2.5 kosong sekali jalan? (data latih)",
      f"Tiap batang = jumlah kejadian kosong. {(rentang_hilang <= 3).mean():.0%} kejadian hanya berlangsung 1 sampai 3 jam.")
ax.set(xlabel="Lama kosong berturut-turut", ylabel="Jumlah kejadian (9 stasiun digabung)", ylim=(0, hitung.max() * 1.12))
simpan_gambar("t2_06_lama_nilai_hilang.png")
print(f"{(rentang_hilang <= 3).mean():.0%} kejadian kosong hanya berlangsung 1 sampai 3 jam.")
print(f"Total rentang nilai hilang: {len(rentang_hilang)} | median {rentang_hilang.median():.0f} jam | terpanjang {rentang_hilang.max()} jam")
for k in [1, 3, 6]:
    print(f"  Rentang <= {k} jam: {(rentang_hilang <= k).mean():.0%} dari seluruh rentang, "
          f"{rentang_hilang[rentang_hilang <= k].sum() / rentang_hilang.sum():.0%} dari jam hilang")
hitung.rename("jumlah kejadian").to_frame()

### 02.8 Apa yang berkaitan dengan PM2.5 besok?
Tabel ini menghitung korelasi antara nilai **sekarang** (jam *t*) dengan PM2.5 **24 jam kemudian** di stasiun yang sama (digeser per stasiun, lalu digabung).
Informasi ini yang benar-benar tersedia saat prediksi dibuat, dan menjadi dasar memilih fitur "kondisi sekarang".

In [ ]:
target_besok = latih.groupby("station")["PM2.5"].shift(-HORIZON)        # PM2.5 pada jam t+24, di stasiun yang sama
korelasi = latih[KOLOM_ANGKA].corrwith(target_besok).sort_values()

korelasi.round(3)

### 02.9 Seberapa jauh masa lalu masih berguna?
Korelasi PM2.5 dengan PM2.5 beberapa jam sebelumnya. Ini membantu memilih *lag* yang dipakai sebagai fitur.

In [ ]:
jarak = [24, 25, 26, 27, 30, 36, 48, 72, 96, 120, 144, 168]   # selisih jam antara dua nilai PM2.5 yang dibandingkan
kor_lag = [latih["PM2.5"].corr(latih.groupby("station")["PM2.5"].shift(j)) for j in jarak]

pd.Series(kor_lag, index=jarak, name="korelasi").round(3).to_frame().T

### 02.10 Perbandingan antarstasiun
Dasar untuk tantangan wajib ketiga: model diuji di stasiun yang belum pernah dilihat. Seberapa beda stasiunnya?
Tabel di bawah memakai data periode latih dari **semua 12 stasiun**, hanya untuk menggambarkan pergeseran distribusi. Model tidak melihat baris stasiun uji.

In [ ]:
ringkas = (tab_dist[["rata_rata", "median", "hilang_pct", "kelompok"]]
           .rename(columns={"hilang_pct": "persen_hilang"}).sort_values("rata_rata"))
ringkas.round(1).to_csv(os.path.join(TABEL, "t2_tabel_ringkas_stasiun.csv"))
ringkas.round(1)

## 03. Pembagian data dan skema validasi
Aturan ujian model, sesuai panduan untuk data berurutan waktu:

1. **Data uji** = 20% periode terakhir (disisihkan di 02.2), untuk 12 stasiun dengan satu batas waktu. Disimpan, dan dibuka **satu kali** di akhir proyek. Dilaporkan terpisah untuk 9 stasiun latih dan 3 stasiun uji.
2. **Data latih** = 80% periode awal dari 9 stasiun latih. Semua pemilihan model dan *tuning* memakai `TimeSeriesSplit(n_splits=5, gap=24)` di sini.
3. **Stasiun uji** tidak dipakai untuk melatih, memvalidasi, maupun menyetel apa pun.
4. Data **tidak diacak**. Setiap *fold* melatih dengan masa lalu dan memvalidasi dengan periode sesudahnya.
5. ***Gap* 24 jam** (`gap=24`). Target sebuah baris adalah PM2.5 24 jam sesudahnya. Tanpa *gap*, target baris latih terakhir jatuh di periode validasi, padahal nilai itu juga muncul sebagai fitur *lag* di data validasi. *Gap* 24 jam menutup celah ini.
6. **Setiap *fold* dibuat dari jam unik, bukan dari baris.** Sembilan stasiun punya jam yang sama. Bila baris gabungan dipotong langsung, satu jam yang sama bisa jatuh di data latih (stasiun A) dan di data validasi (stasiun B), padahal PM2.5 antarstasiun berkorelasi tinggi pada jam yang sama (02.2). Dengan *fold* dari jam unik, semua stasiun mengikuti *fold* jam yang sama.

Sel di bawah membuat *fold* dan memeriksa aturan nomor 4, 5, dan 6.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# Jam unik yang targetnya (t+24) masih di periode latih. Sama dengan jam baris yang dipakai model.
jam_cv = pd.DatetimeIndex(np.sort(latih["waktu"].unique()))
jam_cv = jam_cv[jam_cv < BATAS - pd.Timedelta(hours=HORIZON)]
cv = TimeSeriesSplit(n_splits=5, gap=24)          # jam_cv berurutan per jam, jadi gap=24 berarti 24 jam
fold_cv = [(jam_cv[a], jam_cv[b]) for a, b in cv.split(jam_cv)]

for i, (a, b) in enumerate(fold_cv, 1):
    assert a.max() + pd.Timedelta(hours=HORIZON) < b.min()       # target latih terakhir jatuh sebelum validasi dimulai
    assert len(a.intersection(b)) == 0                           # tidak ada jam yang sama di latih dan validasi
    print(f"Fold {i}: latih s.d. {a[-1]:%Y-%m-%d} | validasi {b[0]:%Y-%m-%d} s.d. {b[-1]:%Y-%m-%d} "
          f"| {len(a):,} jam latih, {len(b):,} jam validasi (x {len(STASIUN_LATIH)} stasiun)")
print("Data uji mulai", BATAS, "(tidak dipakai di fold mana pun)")

In [ ]:
# Peta fold: hanya waktu, tidak ada nilai PM2.5
fig, ax = plt.subplots(figsize=(9, 3.9))
t = mdates.date2num
for i, (a, b) in enumerate(fold_cv):
    ax.barh(i, t(a[-1]) - t(a[0]), left=t(a[0]), color=BIRU, height=0.6)
    ax.barh(i, t(b[-1]) - t(b[0]), left=t(b[0]), color=ORANYE, height=0.6)
ax.barh(len(fold_cv), t(df["waktu"].max()) - t(BATAS), left=t(BATAS), color=ABU, height=0.6)
ax.set_yticks(range(len(fold_cv) + 1), [f"Fold {i}" for i in range(1, len(fold_cv) + 1)] + ["Data uji"])
ax.invert_yaxis()
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
judul(ax, "Validasi berurutan waktu: latih selalu lebih awal dari validasi",
      "Tiap baris satu fold. Ada gap 24 jam antara latih dan validasi (terlalu sempit untuk terlihat). Data uji tidak dipakai.")
ax.set(xlabel="Waktu")
ax.legend(handles=[Patch(color=BIRU, label="Latih"), Patch(color=ORANYE, label="Validasi"),
                   Patch(color=ABU, label="Data uji (disimpan)")], loc="upper right", frameon=False, ncol=3)
ax.grid(axis="y", visible=False)
simpan_gambar("t3_01_fold_validasi.png")

### 03.1 Mengapa skema ini
| Pilihan | Alasan |
|---|---|
| Uji = 20% jam terakhir, tanpa diacak | Data berurutan waktu. Bila diacak, model memakai informasi dari masa depan. Panduan menetapkan skema ini untuk struktur data seperti ini. |
| Satu batas waktu untuk semua stasiun | PM2.5 antarstasiun berkorelasi tinggi pada jam yang sama. Bila stasiun dipotong sendiri-sendiri, jam yang sama bisa ada di data latih satu stasiun dan di data uji stasiun lain. |
| 9 stasiun latih, 3 stasiun uji | Dua pertanyaan diuji terpisah: apakah model memprediksi baik di stasiun yang dikenal, dan apakah tetap baik di stasiun yang belum pernah dilihat (pergeseran distribusi). Periodenya sama, jadi selisih skor kedua kelompok berasal dari stasiun, bukan dari waktu. |
| `TimeSeriesSplit`, jendela latih melebar | Tiap *fold* melatih dengan semua masa lalu yang ada dan memvalidasi dengan periode sesudahnya, persis seperti pemakaian nyata (memprediksi ke depan). |
| 5 *fold* | Contoh di panduan. Tiap *fold* validasi sekitar enam setengah bulan, bergantian mencakup musim dingin dan musim hangat, jadi satu skor tidak hanya mewakili satu musim. |
| `gap=24` pada jam unik | Target sebuah baris adalah PM2.5 24 jam sesudahnya. Tanpa *gap*, target baris latih terakhir jatuh di periode validasi. |
| Data uji dibuka satu kali | Setiap keputusan (fitur, model, pengaturan) dibuat dari validasi data latih. Skor uji hanya dilaporkan, tidak dipakai untuk menyetel ulang. |

Risiko kebocoran dan penjagaannya:

| Risiko | Penjagaan di *notebook* |
|---|---|
| Fitur memakai nilai masa depan | Fitur hanya sampai jam *t*, tanpa interpolasi, dihitung per stasiun (04.1). Diuji dengan memotong deret di tengah dan membandingkan fitur (04.7) |
| Target latih jatuh di periode uji | Baris latih 24 jam terakhir dibuang, dicek dengan `assert` (04.2) |
| Target latih jatuh di periode validasi | `gap=24` pada jam unik, dicek dengan `assert` (03 dan 04.5) |
| Jam yang sama ada di latih dan validasi lewat stasiun lain | *Fold* dari jam unik, dicek dengan `assert` (03) |
| Stasiun uji ikut terlatih | Hanya stasiun latih yang masuk `fitur_latih`, dicek dengan `assert` (04.2 dan 04.7) |
| Praproses belajar dari data validasi atau uji | Imputer dan *scaler* di dalam *pipeline*, dilatih ulang tiap *fold* (04.3, 04.5) |
| Pemilihan stasiun uji memakai data uji | Hanya data sebelum batas waktu yang dipakai (02.2) |
| Data uji dipakai berulang | Disisihkan di 02.2, dibuka satu kali di tahap 6 (evaluasi final) lewat penjaga di 03.3 |

### 03.2 Keterbatasan skema ini
* **Lima *fold* itu sedikit.** Simpangan baku RMSE antar *fold* besar (lihat 04.5), jadi selisih kecil antar model tidak boleh dianggap nyata.
* **Periode uji hanya sekitar 9,5 bulan** (2016-05-12 sampai 2017-02-28), tanpa Maret sampai awal Mei. Skor uji belum tentu mewakili satu siklus musim penuh.
* **Hanya satu pembagian uji.** Skor uji adalah satu hasil, bukan rata-rata dari beberapa periode.
* **Tiga stasiun uji hanya tiga contoh,** dipilih dari data periode latih (02.2). Selisih skor antara kelompok 9 dan kelompok 3 stasiun bisa dipengaruhi oleh stasiun mana yang terpilih, dan tidak boleh dibaca sebagai perkiraan untuk stasiun mana pun.
* **Stasiun Beijing saling berkorelasi tinggi** pada jam yang sama (angkanya dicetak di 02.2). Sembilan stasiun pada satu jam bukan sembilan sampel yang bebas, dan stasiun uji yang berdekatan dengan stasiun latih lebih mudah diprediksi daripada kota lain yang polanya tidak mirip.
* **Identitas stasiun tidak dipakai sebagai fitur** (keputusan penulis 16 Nov 2026). Stasiun uji tidak punya kategori yang dipelajari model, jadi fitur itu tidak bisa dipakai di sana. Model hanya melihat polutan, cuaca, dan kalender.
* **Beda skor antar *fold* belum diuji penyebabnya.** Musim hanya salah satu dugaan.

In [ ]:
# Pemeriksaan skema: tiga kelompok data tidak saling bocor
assert not set(STASIUN_LATIH) & set(STASIUN_UJI)
assert latih["station"].isin(STASIUN_LATIH).all() and latih["waktu"].max() < BATAS
print("Stasiun latih  :", ", ".join(STASIUN_LATIH))
print("Stasiun uji:", ", ".join(STASIUN_UJI))

### 03.3 Penjaga data uji
Berkas fitur uji hanya boleh dibaca lewat fungsi di bawah (dipakai nanti di tahap 6). Fungsi ini **tidak dijalankan di sini**.

* Pembukaan pertama dicatat di `Hasil/Tabel/log_pembukaan_data_uji.csv`: waktu, nama berkas, *hash* berkas, *hash* konfigurasi model, dan alasan.
* Membaca ulang dengan konfigurasi model yang **sama** diizinkan, supaya *notebook* tetap bisa dijalankan ulang dari awal.
* Membuka lagi dengan konfigurasi yang **berbeda** ditolak. Itu tanda model disetel ulang setelah melihat skor uji.

**Batas penjaga ini:** hanya mencegah pembukaan ganda yang tidak disengaja. Orang yang menghapus baris log atau mengubah fungsi tetap bisa melewatinya, dan fungsi tidak bisa tahu apakah konfigurasi dipilih dari validasi atau dari skor uji. Aturan pemilihan model ada di `Laporan/Protokol Evaluasi Data Uji.md`, dan log ini baru berarti kalau di-*commit* bersama versi kode model yang dibekukan.

In [ ]:
import json, hashlib

LOG_UJI = os.path.join(TABEL, "log_pembukaan_data_uji.csv")
KOLOM_LOG = ["waktu", "berkas", "sha256_berkas", "sha256_konfigurasi", "alasan"]

def _sha(b):
    return hashlib.sha256(b).hexdigest()

def buka_data_uji(jalur, konfigurasi, alasan, log=LOG_UJI):
    """Satu-satunya pintu membaca berkas fitur uji. konfigurasi = dict hiperparameter, daftar fitur, dan praproses model final."""
    nama = os.path.basename(jalur)
    sha_konf = _sha(json.dumps(konfigurasi, sort_keys=True, default=str).encode())
    sha_berkas = _sha(open(jalur, "rb").read())
    riwayat = pd.read_csv(log) if os.path.exists(log) else pd.DataFrame(columns=KOLOM_LOG)
    lalu = riwayat[riwayat["berkas"] == nama]
    if len(lalu) == 0:
        baris = pd.DataFrame([[pd.Timestamp.now().isoformat(timespec="seconds"), nama,
                               sha_berkas, sha_konf, alasan]], columns=KOLOM_LOG)
        pd.concat([riwayat, baris], ignore_index=True).to_csv(log, index=False)
        print(f"Data uji {nama} dibuka untuk pertama kali dan dicatat.")
    elif sha_konf in set(lalu["sha256_konfigurasi"]):
        print(f"Data uji {nama} sudah pernah dibuka dengan konfigurasi yang sama: dibaca ulang untuk reproduksi.")
        if sha_berkas not in set(lalu["sha256_berkas"]):
            print("PERINGATAN: isi berkas berbeda dari saat pertama dibuka (cek versi pustaka atau apakah berkas dibuat ulang).")
    else:
        raise RuntimeError(f"{nama} sudah pernah dibuka dengan konfigurasi model yang lain. "
                           "Menyetel ulang model setelah melihat skor uji tidak diizinkan; laporkan skor pertama apa adanya.")
    return pd.read_csv(jalur, index_col=0, parse_dates=True)

print("Penjaga data uji siap. Belum ada berkas uji yang dibuka." if not os.path.exists(LOG_UJI)
      else "Log pembukaan data uji sudah ada:", LOG_UJI if os.path.exists(LOG_UJI) else "")

## 04. Fitur dan *pipeline* praproses
### 04.1 Siapkan fitur
Setiap baris = satu jam *t* di satu stasiun. **Target** = PM2.5 pada jam *t+24* di stasiun yang sama.
Semua fitur hanya memakai data sampai jam *t*, dan dihitung **per stasiun** (*lag* dan *rolling mean* tidak pernah mengambil nilai dari stasiun lain).

| Kelompok | Isi |
|---|---|
| Kondisi sekarang | polutan dan cuaca pada jam *t* |
| *Lag* | PM2.5 beberapa jam sebelumnya |
| *Rolling mean* | rata-rata PM2.5 6, 24, dan 72 jam terakhir |
| Kalender | jam, hari dalam pekan, bulan (dalam bentuk sin dan cos agar jam 23 dekat dengan jam 0) |
| Angin | arah angin diubah menjadi sin dan cos |

Nama stasiun disimpan sebagai kolom `stasiun` untuk keperluan pelaporan, **bukan sebagai fitur**.

**Nilai hilang [Usulan, bukan aturan dosen]:**
* Target hilang: baris dibuang, tidak diisi tebakan.
* Fitur hilang: **tidak diisi di sini** (tanpa *forward fill*). Semua yang kosong diurus imputer median di dalam *pipeline* (04.3), yang hanya belajar dari data latih tiap *fold*.

Mengapa tanpa *forward fill*? Keputusan 15 Nov 2026 dibuat dari perbandingan pada pilot Tiantan: pengisian 0, 1, 3, 6, 12, dan 24 jam tidak memberi beda yang bisa dibedakan dari *noise* antar *seed*. Di 04.6 perbandingan itu diulang pada 9 stasiun latih. Bila hasilnya menunjukkan beda yang jelas melebihi *noise*, keputusan itu perlu ditinjau.

In [ ]:
FFILL_MAKS = 0      # batas forward fill (jam). 0 = tanpa pengisian (keputusan 15 Nov 2026)
ARAH = {a: i * 22.5 for i, a in enumerate(
    ["N","NNE","NE","ENE","E","ESE","SE","SSE","S","SSW","SW","WSW","W","WNW","NW","NNW"])}

def buat_fitur(data_stasiun):
    # Satu stasiun saja. Semua operasi memakai shift positif atau jendela ke belakang.
    d = data_stasiun.set_index("waktu").sort_index()
    f = pd.DataFrame(index=d.index)

    # Target: PM2.5 asli 24 jam ke depan (tanpa pengisian)
    f["target"] = d["PM2.5"].shift(-HORIZON)

    # Kondisi sekarang. Bila FFILL_MAKS > 0, rentang nilai hilang yang pendek diisi nilai terakhir (hanya melihat ke belakang)
    isi = d[KOLOM_ANGKA].ffill(limit=FFILL_MAKS) if FFILL_MAKS > 0 else d[KOLOM_ANGKA]
    for k in KOLOM_ANGKA:
        f[k] = isi[k]

    # Lag dan rolling mean PM2.5 (shift positif = masa lalu)
    for j in [1, 2, 3, 6, 12, 24, 48]:
        f[f"PM2.5_lag{j}"] = isi["PM2.5"].shift(j)
    for w in [6, 24, 72]:
        f[f"PM2.5_rata{w}j"] = isi["PM2.5"].rolling(w, min_periods=w // 2).mean()

    # Kalender dan arah angin dalam bentuk lingkaran
    wd = d["wd"].ffill(limit=FFILL_MAKS) if FFILL_MAKS > 0 else d["wd"]
    sudut = wd.map(ARAH)
    f["angin_sin"], f["angin_cos"] = np.sin(np.radians(sudut)), np.cos(np.radians(sudut))
    f["jam_sin"], f["jam_cos"] = np.sin(2*np.pi*d.index.hour/24), np.cos(2*np.pi*d.index.hour/24)
    f["hari_sin"], f["hari_cos"] = np.sin(2*np.pi*d.index.dayofweek/7), np.cos(2*np.pi*d.index.dayofweek/7)
    f["bulan_sin"], f["bulan_cos"] = np.sin(2*np.pi*d.index.month/12), np.cos(2*np.pi*d.index.month/12)
    return f

def fitur_stasiun(data, daftar):
    # Fitur dihitung terpisah untuk setiap stasiun, lalu ditumpuk. Kolom `stasiun` bukan fitur.
    bagian = []
    for s in daftar:
        f = buat_fitur(data[data["station"] == s])
        f.insert(0, "stasiun", s)
        bagian.append(f)
    return pd.concat(bagian)

### 04.2 Potong fitur di batas data uji
Fitur dibuat dari deret lengkap tiap stasiun (karena hanya melihat ke belakang, ini aman), lalu dipotong di `BATAS` dan menurut stasiun:

* **Latih:** hanya 9 stasiun latih, hanya baris yang targetnya (jam *t+24*) jatuh sebelum `BATAS`.
* **Uji:** baris jam *t* di `BATAS` atau sesudahnya, dari 12 stasiun, diberi label kelompok (`stasiun_latih` atau `stasiun_uji`).
* **Dibuang:** baris 3 stasiun uji sebelum `BATAS` (tidak dipakai), dan baris 24 jam terakhir sebelum `BATAS` di semua stasiun.

**Celah kebocoran yang ditutup:** baris latih 24 jam terakhir punya target yang jatuh di periode uji.
Baris itu dibuang supaya tidak ada nilai uji yang ikut dilatih.

In [ ]:
fitur_semua = fitur_stasiun(df, STASIUN_LATIH + STASIUN_UJI)
waktu_target = fitur_semua.index + pd.Timedelta(hours=HORIZON)
ada_target = fitur_semua["target"].notna()

fitur_latih = fitur_semua[fitur_semua["stasiun"].isin(STASIUN_LATIH) & (waktu_target < BATAS) & ada_target]
fitur_uji = fitur_semua[(fitur_semua.index >= BATAS) & ada_target].copy()
fitur_uji.insert(1, "kelompok_uji", np.where(fitur_uji["stasiun"].isin(STASIUN_UJI), "stasiun_uji", "stasiun_latih"))
del fitur_semua

KOLOM_FITUR = [c for c in fitur_latih.columns if c not in ("stasiun", "target")]
n_stasiun_uji = (fitur_uji["kelompok_uji"] == "stasiun_uji").sum()
print(f"Fitur latih: {len(fitur_latih):,} baris, {len(KOLOM_FITUR)} fitur, {fitur_latih['stasiun'].nunique()} stasiun")
print(f"Fitur uji  : {len(fitur_uji):,} baris (disimpan, tidak dibuka): "
      f"{len(fitur_uji) - n_stasiun_uji:,} stasiun latih + {n_stasiun_uji:,} stasiun uji")
print("Target latih terakhir:", (fitur_latih.index.max() + pd.Timedelta(hours=HORIZON)),
      "| harus sebelum", BATAS)

assert fitur_latih.index.max() + pd.Timedelta(hours=HORIZON) < BATAS
assert set(fitur_latih["stasiun"]) == set(STASIUN_LATIH)                 # tidak ada baris stasiun uji di data latih
assert fitur_uji.index.min() >= BATAS
assert set(fitur_uji.loc[fitur_uji["kelompok_uji"] == "stasiun_uji", "stasiun"]) == set(STASIUN_UJI)
assert set(fitur_uji.loc[fitur_uji["kelompok_uji"] == "stasiun_latih", "stasiun"]) == set(STASIUN_LATIH)

In [ ]:
# Cek: berapa persen fitur yang masih kosong (tanpa pengisian, diurus imputer di pipeline)
sisa = fitur_latih[KOLOM_FITUR].isna().mean().mul(100)
print(f"Fitur dengan nilai kosong tersisa: {(sisa > 0).sum()} dari {len(sisa)}, "
      f"tertinggi {sisa.idxmax()} = {sisa.max():.2f}%")

### 04.3 *Pipeline* praproses
Langkah yang "belajar" dari data (mengisi kosong dengan median, menyamakan skala) ditaruh di dalam *pipeline*.
*Pipeline* ini dilatih ulang di setiap *fold* `TimeSeriesSplit`, jadi hanya belajar dari data latih *fold* itu.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Belum dilatih di sini. Di tahap 4, pipeline ini dilatih ulang di setiap fold (cv dari bagian 03).
praproses = Pipeline([("isi_kosong", SimpleImputer(strategy="median")),
                      ("skala", StandardScaler())])
praproses

### 04.4 Simpan hasil
Dua berkas CSV: `fitur_latih.csv` (9 stasiun latih, periode latih) dan `fitur_uji.csv` (12 stasiun, periode uji, dengan kolom `kelompok_uji`). Berkas uji hanya disimpan.

In [ ]:
fitur_latih.rename_axis("waktu").to_csv(os.path.join(OLAHAN, "fitur_latih.csv"))
fitur_uji.rename_axis("waktu").to_csv(os.path.join(OLAHAN, "fitur_uji.csv"))
for nama_folder, isi in (("Gambar", GAMBAR), ("Tabel", TABEL), ("Fitur", OLAHAN)):
    print(f"{nama_folder}:", ", ".join(sorted(os.listdir(isi))))

### 04.5 *Baseline* (hasil *cross-validation*)
*Baseline* adalah titik acuan. Tanpanya, sebuah angka RMSE tidak bermakna.

*Baseline* yang dipakai satu: ***naive forecast***, PM2.5 24 jam lagi dianggap sama dengan PM2.5 sekarang, di stasiun yang sama. Ini *baseline* wajib untuk deret waktu **[Panduan]**.

Aturan evaluasi:
* `TimeSeriesSplit(n_splits=5, gap=24)` pada jam unik data latih (03). Data uji dan stasiun uji tidak disentuh.
* Skor tiap *fold* dihitung pada baris validasi dari 9 stasiun digabung.
* *Naive forecast* butuh PM2.5 jam *t*. Baris validasi yang PM2.5 jam *t*-nya kosong dibuang, karena *naive forecast* tidak bisa memprediksi tanpa nilai itu.
* Hasil dilaporkan sebagai rata-rata ± simpangan baku antar *fold*. RMSE jadi metrik utama, MAE pendamping.

In [ ]:
X = fitur_latih[KOLOM_FITUR]
y = fitur_latih["target"]

def rmse(y_asli, y_tebak): return float(np.sqrt(np.mean((y_asli - y_tebak) ** 2)))
def mae(y_asli, y_tebak):  return float(np.mean(np.abs(y_asli - y_tebak)))

nama_model = ["Naive forecast"]
skor = {n: {"RMSE": [], "MAE": []} for n in nama_model}
dibuang = jumlah_validasi = 0
for jam_a, jam_b in fold_cv:
    # target baris latih harus jatuh sebelum periode validasi dimulai, dan tidak ada jam yang dipakai bersama
    assert jam_a.max() + pd.Timedelta(hours=HORIZON) < jam_b.min()
    a, b = X.index.isin(jam_a), X.index.isin(jam_b)
    ada = X[b]["PM2.5"].notna().to_numpy()           # naive butuh PM2.5 jam t
    dibuang += int((~ada).sum()); jumlah_validasi += int(b.sum())
    Xv, yv = X[b][ada], y[b][ada].to_numpy()

    tebak = {"Naive forecast": Xv["PM2.5"].to_numpy()}

    for n in nama_model:
        skor[n]["RMSE"].append(rmse(yv, tebak[n]))
        skor[n]["MAE"].append(mae(yv, tebak[n]))

tabel_baseline = pd.DataFrame({
    "RMSE rata-rata": [np.mean(skor[n]["RMSE"]) for n in nama_model],
    "RMSE sd":        [np.std(skor[n]["RMSE"], ddof=1) for n in nama_model],
    "MAE rata-rata":  [np.mean(skor[n]["MAE"]) for n in nama_model],
    "MAE sd":         [np.std(skor[n]["MAE"], ddof=1) for n in nama_model]}, index=nama_model).round(2)
tabel_baseline.to_csv(os.path.join(TABEL, "t4_tabel_baseline.csv"))
fig, ax = plt.subplots()
rm = skor["Naive forecast"]["RMSE"]
ax.bar([f"Fold {i}" for i in range(1, len(rm) + 1)], rm, color=BIRU, width=0.55)
for x, v in enumerate(rm):
    ax.text(x, v, f"{v:.0f}", ha="center", va="bottom", fontsize=9)
judul(ax, "Galat baseline naive forecast di tiap fold validasi (data latih)",
      "RMSE: makin kecil makin baik. Naive = tebakan PM2.5 besok sama dengan sekarang. Data uji tidak dipakai.")
ax.set(xlabel="Fold validasi", ylabel="RMSE (µg/m³)", ylim=(0, max(rm) * 1.15))
ax.grid(axis="x", visible=False)
simpan_gambar("t4_01_baseline_rmse_per_fold.png")
print(f"Baris validasi dibuang (PM2.5 jam t kosong): {dibuang} dari {jumlah_validasi:,}")
print("RMSE per fold:")
for n in nama_model:
    print(f"  {n:<15}", np.round(skor[n]["RMSE"], 1))
tabel_baseline

### 04.6 Dasar keputusan: batas *forward fill*
*Bagian pendukung: boleh dilewati saat membaca pertama.* Isinya hanya pembenaran satu keputusan (tanpa *forward fill*), tidak dipakai bagian lain.

Keputusan 15 Nov 2026: tanpa *forward fill*, diambil dari pilot Tiantan. Di sini dasarnya dihitung ulang pada **9 stasiun latih**, **hanya dari data latih** (data uji dan stasiun uji tidak dipakai). Fitur dibuat ulang per stasiun dengan batas pengisian 0, 1, 3, 6, 12, dan 24 jam, lalu dua model dinilai dengan *fold* jam unik yang sama dengan bagian 03.
Karena rentang nilai hilang yang pendek diisi nilai terakhir (hanya melihat ke belakang), tidak ada nilai masa depan yang masuk.

Yang dibandingkan adalah selisih antar pengaturan *dan* *noise* antar *seed* pada pengaturan yang sama. Selisih yang lebih kecil dari *noise* tidak boleh dianggap nyata.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

def fitur_dengan_ffill(ff):
    # Dihitung dari `latih` (9 stasiun latih, sebelum BATAS) saja. Target 24 jam terakhir otomatis kosong, jadi dibuang.
    global FFILL_MAKS
    simpan, FFILL_MAKS = FFILL_MAKS, ff
    try:
        return fitur_stasiun(latih, STASIUN_LATIH)
    finally:
        FFILL_MAKS = simpan

def model_uji(nama, seed=SEED):
    reg = Ridge(alpha=1.0) if nama == "Ridge" else HistGradientBoostingRegressor(
        max_iter=200, learning_rate=0.1, max_depth=4, random_state=seed)
    return Pipeline([("isi", SimpleImputer(strategy="median")), ("skala", StandardScaler()), ("m", reg)])

def baris_fold(F, jam):
    x = F[F.index.isin(jam)].dropna(subset=["target"])
    return x[KOLOM_FITUR], x["target"]

daftar_ff = [0, 1, 3, 6, 12, 24]
skor_ff = {}
for ff in daftar_ff:
    F = fitur_dengan_ffill(ff)
    for nm in ["Ridge", "HGB"]:
        r = []
        for a, b in fold_cv:
            Xa, ya = baris_fold(F, a); Xb, yb = baris_fold(F, b)
            r.append(rmse(yb, model_uji(nm).fit(Xa, ya).predict(Xb)))
        skor_ff[(ff, nm)] = np.array(r)

baris_tabel = []
for nm in ["Ridge", "HGB"]:
    dasar = skor_ff[(3, nm)]
    for ff in daftar_ff:
        r = skor_ff[(ff, nm)]
        baris_tabel.append({"model": nm, "ffill_maks_jam": ff, "RMSE rata-rata": r.mean(), "RMSE sd antar fold": r.std(ddof=1),
                            "selisih vs 3 jam": (r - dasar).mean(), "fold lebih baik dari 3 jam": int(((r - dasar) < 0).sum())})
tabel_ff = pd.DataFrame(baris_tabel).round(3)
tabel_ff.to_csv(os.path.join(TABEL, "t2_tabel_ffill.csv"), index=False)
tabel_ff

In [ ]:
# Tingkat noise dasar: pengaturan yang sama (ffill = 3 jam), hanya seed HGB yang diganti
F3 = fitur_dengan_ffill(3)
rata_seed = []
for sd in range(6):
    r = []
    for a, b in fold_cv:
        Xa, ya = baris_fold(F3, a); Xb, yb = baris_fold(F3, b)
        r.append(rmse(yb, model_uji("HGB", seed=sd).fit(Xa, ya).predict(Xb)))
    rata_seed.append(np.mean(r))
rentang_seed = max(rata_seed) - min(rata_seed)
rentang_ff = {nm: float(np.ptp([skor_ff[(ff, nm)].mean() for ff in daftar_ff])) for nm in ["Ridge", "HGB"]}
sd_fold = {nm: float(np.mean([skor_ff[(ff, nm)].std(ddof=1) for ff in daftar_ff])) for nm in ["Ridge", "HGB"]}
print("RMSE rata-rata per seed HGB (ffill = 3 jam):", np.round(rata_seed, 2), "| rentang", round(rentang_seed, 2))
print("Rentang rata-rata RMSE antar pengaturan ffill :", {k: round(v, 2) for k, v in rentang_ff.items()})
print("Simpangan baku rata-rata antar fold           :", {k: round(v, 2) for k, v in sd_fold.items()})
print("Baca: rentang antar pengaturan yang tidak melebihi simpangan baku antar fold (dan, untuk HGB, rentang seed) "
      "tidak boleh dianggap beda nyata.")

### 04.7 Pemeriksaan kesesuaian dengan desain 9/3
Sel di bawah memeriksa desain pada data yang benar-benar dibuat di atas. Semua pemeriksaan memakai `assert`, jadi *notebook* berhenti bila ada yang dilanggar.

1. Stasiun latih dan stasiun uji tidak beririsan, jumlahnya 9 dan 3.
2. Berkas latih hanya berisi stasiun latih, dan target barisnya jatuh sebelum `BATAS`.
3. Berkas uji berisi jam sejak `BATAS`, untuk dua kelompok, dan stasiun uji **hanya** ada di berkas uji.
4. *Fold* validasi: *gap* minimal 24 jam dan tidak ada jam yang sama di latih dan validasi.
5. **Fitur tidak melihat masa depan:** fitur satu stasiun dihitung dari deret yang dipotong di tengah, lalu dibandingkan dengan fitur dari deret penuh pada jam yang sama. Keduanya harus sama persis (target dikecualikan, karena target memang melihat ke depan).
6. Status penjaga data uji.

In [ ]:
# 1
assert len(STASIUN_LATIH) == 9 and len(STASIUN_UJI) == 3 and not set(STASIUN_LATIH) & set(STASIUN_UJI)
# 2
assert set(fitur_latih["stasiun"]) == set(STASIUN_LATIH)
assert (fitur_latih.index + pd.Timedelta(hours=HORIZON) < BATAS).all() and fitur_latih["target"].notna().all()
# 3
assert (fitur_uji.index >= BATAS).all() and fitur_uji["target"].notna().all()
assert set(fitur_uji["kelompok_uji"]) == {"stasiun_latih", "stasiun_uji"}
assert not set(fitur_latih["stasiun"]) & set(fitur_uji.loc[fitur_uji["kelompok_uji"] == "stasiun_uji", "stasiun"])
# 4
for a, b in fold_cv:
    assert a.max() + pd.Timedelta(hours=HORIZON) < b.min() and len(a.intersection(b)) == 0
# 5
T0 = pd.Timestamp("2015-06-01 00:00")
for s in (STASIUN_LATIH[0], STASIUN_UJI[0]):
    penuh = buat_fitur(df[df["station"] == s]).loc[:T0, KOLOM_FITUR]
    potong = buat_fitur(df[(df["station"] == s) & (df["waktu"] <= T0)]).loc[:T0, KOLOM_FITUR]
    pd.testing.assert_frame_equal(penuh, potong)
    print(f"Fitur {s} sampai {T0} identik dengan atau tanpa data sesudahnya ({len(penuh):,} baris diperiksa).")
# 6
print("Penjaga data uji:", "BELUM ADA berkas uji yang dibuka." if not os.path.exists(LOG_UJI)
      else f"log pembukaan ada di {LOG_UJI}")
print("Semua pemeriksaan desain 9/3 lolos.")

# Ringkasan visual. Baris 1 sampai 5 sudah lolos (kalau tidak, notebook berhenti di atas), baris 6 dicek sekarang.
uji_belum_dibuka = not os.path.exists(LOG_UJI)
hasil = [
    (True, "Stasiun latih (9) dan stasiun uji (3) tidak beririsan"),
    (True, "Berkas latih hanya berisi stasiun latih, target jatuh sebelum batas data uji"),
    (True, "Berkas uji mulai dari batas, stasiun uji hanya ada di berkas uji"),
    (True, "Fold validasi punya gap 24 jam dan tidak berbagi jam"),
    (True, "Fitur tidak melihat masa depan (sama persis dengan atau tanpa data sesudahnya)"),
    (uji_belum_dibuka, "Data uji belum dibuka (tidak ada log pembukaan)" if uji_belum_dibuka else "Data uji SUDAH dibuka (ada log pembukaan)"),
]
fig, ax = plt.subplots(figsize=(9, 3.4))
for i, (ok, teks) in enumerate(hasil):
    y = len(hasil) - i
    ax.text(0.0, y, "✓ LOLOS" if ok else "! CEK", fontsize=10, fontweight="bold", va="center", color="#1f7a3d" if ok else ORANYE)
    ax.text(0.14, y, teks, fontsize=10, va="center")
ax.set(xlim=(0, 1), ylim=(0.4, len(hasil) + 0.6))
ax.axis("off")
ax.set_title(f"Pemeriksaan desain 9/3: {sum(ok for ok, _ in hasil)} dari {len(hasil)} lolos", loc="left", fontsize=12, fontweight="bold", pad=28)
ax.text(0, len(hasil) + 0.9, "Pemeriksaan memakai assert, jadi notebook berhenti bila ada yang dilanggar.", fontsize=9, color="#52514e", va="bottom")
simpan_gambar("t4_02_pemeriksaan_desain.png")

## 05. Model lanjutan dan *tuning* (hasil *cross-validation*)
Tahap 5 memakai aturan evaluasi yang sama dengan *baseline* di 04.5, supaya setiap model dibandingkan dengan *naive forecast* pada *fold* dan baris yang sama:

* `TimeSeriesSplit(n_splits=5, gap=24)` pada jam unik data latih 9 stasiun (bagian 03). **Data uji dan stasiun uji tidak disentuh.** Data uji baru dibuka satu kali di tahap 6 (evaluasi final), setelah satu model final dipilih dari tabel di bagian ini.
* Setiap model dilatih ulang di tiap *fold* lewat *pipeline*: imputer median dan *scaler* hanya belajar dari data latih *fold* itu.
* Skor dihitung pada baris validasi yang sama dengan *naive*, yaitu baris yang PM2.5 jam *t*-nya tidak kosong. Sel di bawah memeriksa dengan `assert` bahwa galat *naive* di sini sama persis dengan 04.5.
* Dilaporkan rata-rata ± simpangan baku antar *fold*: RMSE sebagai metrik utama, MAE sebagai pendamping.

**Kandidat.** Pilihan keluarga model adalah keputusan penulis (22 Nov 2026). Pengaturan *gradient boosting* dan `l1_ratio` adalah usulan Claude: mengikuti pengaturan di 04.6 dan nilai bawaan *scikit-learn*.

| Model | Pengaturan | Alasan |
|---|---|---|
| Ridge | `alpha=1` | Regresi linear dengan penalti L2. Pembanding linear yang kuat bila fitur *lag* sudah memuat sebagian besar informasi. |
| Elastic Net | `alpha` 0,01; 0,1; 1 dan `l1_ratio=0,5` | Penalti L1 dan L2. Tiga nilai `alpha` adalah *tuning* satu model, bukan tiga model. |
| *Gradient boosting* | `HistGradientBoostingRegressor`, 200 iterasi, *learning rate* 0,1, kedalaman 4, tanpa *early stopping* | Model pohon, dapat menangkap hubungan nonlinear antar fitur. |
| *Gradient boosting* (*early stopping* bawaan) | Sama, dengan `early_stopping=True` | Memeriksa apakah *early stopping* bawaan mengubah hasil. |

Jadi ada tiga keluarga model (linear L2, linear L1 dan L2, *boosting*) dalam enam konfigurasi.

In [ ]:
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import HistGradientBoostingRegressor

L1_RATIO = 0.5          # Elastic Net: campuran L1 dan L2 (nilai bawaan scikit-learn)

def pipa(estimator):
    # Imputer median dan scaler ada di dalam pipeline, jadi dilatih ulang pada data latih setiap fold
    return Pipeline([("isi_kosong", SimpleImputer(strategy="median")),
                     ("skala", StandardScaler()),
                     ("model", estimator)])

# Urutan dari yang paling sederhana ke paling kompleks [Usulan]. Dipakai aturan pemilihan di bawah.
KANDIDAT = {
    "Ridge (alpha=1)": lambda: Ridge(alpha=1.0),
    "Elastic Net (alpha=1)": lambda: ElasticNet(alpha=1.0, l1_ratio=L1_RATIO, max_iter=5000, random_state=SEED),
    "Elastic Net (alpha=0,1)": lambda: ElasticNet(alpha=0.1, l1_ratio=L1_RATIO, max_iter=5000, random_state=SEED),
    "Elastic Net (alpha=0,01)": lambda: ElasticNet(alpha=0.01, l1_ratio=L1_RATIO, max_iter=5000, random_state=SEED),
    "Gradient boosting": lambda: HistGradientBoostingRegressor(
        max_iter=200, learning_rate=0.1, max_depth=4, early_stopping=False, random_state=SEED),
    "Gradient boosting (early stopping bawaan)": lambda: HistGradientBoostingRegressor(
        max_iter=200, learning_rate=0.1, max_depth=4, early_stopping=True, random_state=SEED),
}

# Dibuat ulang dari fitur_latih (nama X dan y bisa tertimpa sel lain di antara 04.5 dan sini)
X5 = fitur_latih[KOLOM_FITUR]
y5 = fitur_latih["target"]

skor5 = {n: {"RMSE": [], "MAE": []} for n in KANDIDAT}
iterasi_es = []                      # jumlah iterasi yang dipakai early stopping bawaan, per fold
for k, (jam_a, jam_b) in enumerate(fold_cv):
    # Aturan sama dengan 04.5: target latih jatuh sebelum validasi, baris validasi = baris yang dipakai naive
    assert jam_a.max() + pd.Timedelta(hours=HORIZON) < jam_b.min()
    a, b = X5.index.isin(jam_a), X5.index.isin(jam_b)
    ada = X5[b]["PM2.5"].notna().to_numpy()
    Xa, ya = X5[a], y5[a]
    Xv, yv = X5[b][ada], y5[b][ada].to_numpy()
    # Pemeriksaan: baris dan galat naive di sini identik dengan 04.5, jadi perbandingannya apel dengan apel
    assert abs(rmse(yv, Xv["PM2.5"].to_numpy()) - skor["Naive forecast"]["RMSE"][k]) < 1e-9

    for nama, buat in KANDIDAT.items():
        model = pipa(buat()).fit(Xa, ya)                 # hanya melihat data latih fold ini
        tebak = model.predict(Xv)
        skor5[nama]["RMSE"].append(rmse(yv, tebak))
        skor5[nama]["MAE"].append(mae(yv, tebak))
        if nama.endswith("(early stopping bawaan)"):
            iterasi_es.append(int(model.named_steps["model"].n_iter_))
    print(f"Fold {k + 1} selesai", flush=True)
print("Iterasi yang dipakai early stopping bawaan per fold (maksimum 200):", iterasi_es)

### 05.2 Tabel perbandingan
Satu tabel untuk *naive* dan keenam konfigurasi. Kolom "Fold lebih baik dari naive" menghitung di berapa *fold* RMSE model lebih kecil daripada RMSE *naive* pada *fold* yang sama.

In [ ]:
rmse_naive = np.array(skor["Naive forecast"]["RMSE"])
mae_naive = np.array(skor["Naive forecast"]["MAE"])
baris = [{"Model": "Naive forecast", "RMSE rata-rata": rmse_naive.mean(), "RMSE sd": rmse_naive.std(ddof=1),
          "MAE rata-rata": mae_naive.mean(), "MAE sd": mae_naive.std(ddof=1),
          "Perbaikan RMSE vs naive (%)": 0.0, "Fold lebih baik dari naive": "n/a"}]
for n in KANDIDAT:
    r, m = np.array(skor5[n]["RMSE"]), np.array(skor5[n]["MAE"])
    baris.append({"Model": n, "RMSE rata-rata": r.mean(), "RMSE sd": r.std(ddof=1),
                  "MAE rata-rata": m.mean(), "MAE sd": m.std(ddof=1),
                  "Perbaikan RMSE vs naive (%)": 100 * (rmse_naive.mean() - r.mean()) / rmse_naive.mean(),
                  "Fold lebih baik dari naive": f"{int((r < rmse_naive).sum())} dari {len(r)}"})
tabel5 = pd.DataFrame(baris).set_index("Model")
tabel5.round(2).to_csv(os.path.join(TABEL, "t5_tabel_model_lanjutan.csv"))
tabel_fold5 = pd.DataFrame({"Naive forecast": rmse_naive, **{n: skor5[n]["RMSE"] for n in KANDIDAT}},
                           index=[f"Fold {i}" for i in range(1, len(rmse_naive) + 1)]).round(2)
tabel_fold5.to_csv(os.path.join(TABEL, "t5_tabel_rmse_per_fold.csv"))
print(tabel_fold5.T.to_string())
tabel5.round(2)

### 05.3 Aturan pemilihan dan cara membaca tabel
Aturan memakai butir 2 di `Laporan/Protokol Evaluasi Data Uji.md` (**[Usulan]**, menunggu persetujuan penulis). Aturan itu ditulis sebelum hasil dilihat dan tidak diubah sesudahnya:

1. Dasar pemilihan hanya rata-rata RMSE validasi silang 5 *fold*.
2. Bila selisih rata-rata RMSE dengan model terbaik lebih kecil dari satu simpangan baku antar *fold* (simpangan baku model terbaik), kedua model dianggap tidak terbedakan, dan yang lebih sederhana dipilih. Urutan kesederhanaan (**[Usulan]**) mengikuti urutan di tabel: Ridge, Elastic Net dari `alpha` besar ke kecil, lalu *gradient boosting*.
3. Model final harus lebih baik dari *naive forecast* di validasi. Bila tidak, laporkan apa adanya.

Dua pemeriksaan tambahan: selisih RMSE terhadap Ridge pada *fold* yang sama (perbandingan berpasangan, tidak ikut kesulitan musim), dan apakah `alpha` terbaik Elastic Net ada di tepi grid. Hasil aturan ini hanya **kandidat**. Konfigurasi belum dibekukan dan data uji belum dibuka.

In [ ]:
# Aturan pemilihan (Protokol Evaluasi Data Uji, butir 2, [Usulan]): ditulis sebelum hasil dilihat, tidak diubah setelahnya.
#  1) Dasar: rata-rata RMSE validasi silang 5 fold.
#  2) Bila selisih rata-rata RMSE dengan model terbaik lebih kecil dari satu simpangan baku antar fold
#     (simpangan baku model terbaik), dua model dianggap tidak terbedakan dan yang lebih sederhana dipilih.
#  3) Model final harus lebih baik dari naive forecast.
rm5 = tabel5.loc[list(KANDIDAT), "RMSE rata-rata"]
terbaik = rm5.idxmin()
sd_terbaik = tabel5.loc[terbaik, "RMSE sd"]
setara = [n for n in KANDIDAT if rm5[n] - rm5[terbaik] < sd_terbaik]       # KANDIDAT sudah urut dari yang paling sederhana
pilih = setara[0]
print(f"RMSE terendah     : {terbaik} ({rm5[terbaik]:.2f}); simpangan baku antar fold {sd_terbaik:.2f}")
print(f"Tidak terbedakan  : {', '.join(setara)}")
print(f"Menurut aturan    : {pilih} ({rm5[pilih]:.2f}), lebih baik dari naive ({rmse_naive.mean():.2f}): {rm5[pilih] < rmse_naive.mean()}")
print("Ini kandidat, belum model final: konfigurasi belum dibekukan dan data uji belum dibuka.")

# Pemeriksaan tambahan 1: perbandingan berpasangan terhadap Ridge pada fold yang sama
# (selisih per fold tidak ikut kesulitan musim, jadi lebih tajam daripada membandingkan simpangan baku mentah)
dasar_r = np.array(skor5["Ridge (alpha=1)"]["RMSE"])
pasang = []
for n in KANDIDAT:
    d = np.array(skor5[n]["RMSE"]) - dasar_r
    pasang.append({"Model": n, "Selisih RMSE vs Ridge, rata-rata": d.mean(), "sd selisih antar fold": d.std(ddof=1),
                   "Fold lebih baik dari Ridge": f"{int((d < 0).sum())} dari {len(d)}"})
tabel_pasang = pd.DataFrame(pasang).set_index("Model").drop(index="Ridge (alpha=1)").round(2)
tabel_pasang.to_csv(os.path.join(TABEL, "t5_tabel_selisih_vs_ridge.csv"))
print("\nSelisih RMSE terhadap Ridge per fold (negatif = lebih baik dari Ridge):")
print(tabel_pasang.to_string())

# Pemeriksaan tambahan 2: alpha terbaik Elastic Net ada di tepi grid?
en = [n for n in KANDIDAT if n.startswith("Elastic Net")]
a_en = {n: float(n.split("alpha=")[1].rstrip(")").replace(",", ".")) for n in en}
en_terbaik = min(en, key=lambda n: rm5[n])
if a_en[en_terbaik] in (min(a_en.values()), max(a_en.values())):
    print(f"\nCatatan: alpha terbaik Elastic Net ({a_en[en_terbaik]}) ada di tepi grid {sorted(a_en.values())}. "
          "Optimum bisa di luar grid, dan selisih antar alpha ({:.2f}) jauh di bawah simpangan baku antar fold.".format(
              rm5[en].max() - rm5[en].min()))

# Grafik: rata-rata RMSE ± simpangan baku, titik = nilai tiap fold
fig, ax = plt.subplots(figsize=(9, 4.4))
urut = ["Naive forecast"] + list(KANDIDAT)
for i, n in enumerate(urut[::-1]):
    nilai = rmse_naive if n == "Naive forecast" else np.array(skor5[n]["RMSE"])
    warna = ABU if n == "Naive forecast" else (ORANYE if n.startswith("Gradient") else BIRU)
    ax.barh(i, nilai.mean(), xerr=nilai.std(ddof=1), color=warna, height=0.55, alpha=0.9,
            error_kw={"ecolor": "#2c2c2a", "capsize": 3, "lw": 1})
    ax.scatter(nilai, [i] * len(nilai), s=14, color="#2c2c2a", zorder=3)
    ax.text(0.8, i, f"{nilai.mean():.1f}", va="center", ha="left", color="white", fontsize=9, fontweight="bold")
ax.axvline(rmse_naive.mean(), color="#2c2c2a", ls="--", lw=1)
ax.set_yticks(range(len(urut)))
ax.set_yticklabels(urut[::-1])
judul(ax, "RMSE validasi silang: naive dan model lanjutan (9 stasiun latih)",
      "Batang = rata-rata 5 fold, garis = ± sd, titik = tiap fold, putus-putus = rata-rata naive")
ax.set(xlabel="RMSE (µg/m³), makin kecil makin baik")
ax.grid(axis="y", visible=False)
simpan_gambar("t5_01_rmse_model_lanjutan.png")

### 05.4 Keterbatasan dan yang belum dikerjakan
* **Lima *fold* itu sedikit.** Simpangan baku RMSE antar *fold* besar dan sebagian besar berasal dari beda kesulitan antar periode, bukan dari beda antar model. Selisih kecil tidak boleh dianggap nyata.
* ***Tuning* belum sistematis.** Grid `alpha` Elastic Net hanya tiga nilai dan `l1_ratio` tetap. *Gradient boosting* belum dicari *hyperparameter*-nya (kedalaman, *learning rate*, jumlah iterasi, regularisasi L2). Bagian ini baru perbandingan awal. Pencarian lanjutan harus memakai *fold* yang sama dan tetap tanpa data uji.
* ***Early stopping* bawaan `HistGradientBoostingRegressor`** memisahkan data validasi internal secara acak dari data latih *fold* (bukan menurut waktu). Itu tidak membocorkan data validasi *fold*, tetapi tidak mengikuti urutan waktu. Jumlah iterasi yang tercetak di atas menunjukkan apakah ia sempat berhenti sebelum 200. Bila selalu 200, varian ini praktis hanya melatih model pada sebagian data latih.
* **Pemilihan dari enam konfigurasi dengan validasi yang sama** membuat skor validasi model terpilih sedikit optimistis. Angka di tabel ini tidak boleh dilaporkan sebagai kinerja akhir. Kinerja akhir hanya dari data uji, satu kali.
* **Skor hanya untuk baris dengan PM2.5 jam *t* tidak kosong** (sekitar 2% baris validasi dibuang, mengikuti *naive*). Skor model bersyarat pada hal itu.
* **Identitas stasiun bukan fitur** (keputusan 16 Nov 2026), jadi hasil ini tidak menjawab apakah fitur itu membantu di stasiun latih.

### 05.5 Eksperimen lanjutan: fitur, model, *tuning*, dan transformasi target
Pertanyaan: adakah cara menurunkan RMSE validasi dari sekitar 72,5 (model terbaik di 05.1) tanpa melanggar aturan validasi? Semua eksperimen memakai *fold*, baris validasi, dan praproses per *fold* yang sama dengan 04.5 dan 05.1 (sel di bawah memeriksa dengan `assert` bahwa galat *naive* identik). **Data uji tidak dibuka.**

| Langkah | Isi | Anggaran |
|---|---|---|
| 1 | Ablasi fitur: Ridge dan Elastic Net (`alpha=0,1`) pada lima himpunan fitur | 10 konfigurasi |
| 2 | Keluarga model pada F1, F2, dan F3: *gradient boosting* (pengaturan 04.6), Ridge lalu *boosting* pada residual, rata-rata Ridge dan *boosting*, *random forest* | 11 konfigurasi |
| 3 | *Tuning* acak pada F2 dan F3: 12 konfigurasi *gradient boosting* (jumlah iterasi dipilih dari kurva rata-rata 5 *fold*) dan 6 konfigurasi *random forest* per himpunan fitur | 36 konfigurasi |
| 4 | Rata-rata sederhana (bobot sama, tidak disetel) dari model terpilih di langkah 3 | 8 konfigurasi |
| 5 | Transformasi target (akar kuadrat dan `log1p`) untuk Ridge dan *boosting* terpilih | 8 konfigurasi |
| 6 | Putaran 2: grid *random forest* diperluas ke arah regularisasi lebih kuat, ditambah *Extra Trees*, hanya F3 | 9 konfigurasi + 3 rata-rata |

**Catatan keterbukaan:** langkah 1 dan 2 dirancang sebelum dijalankan. Langkah 3 sampai 6 ditentukan setelah melihat hasil langkah sebelumnya (misalnya langkah 6 karena konfigurasi *random forest* terbaik ada di tepi grid). Semua konfigurasi yang dijalankan dicatat, termasuk yang gagal: rata-rata dan simpangan baku semua konfigurasi *tuning* di `t5_tabel_tuning.csv` dan `t5_tabel_tuning_putaran2.csv`, nilai tiap *fold* untuk 53 eksperimen (termasuk konfigurasi terbaik hasil *tuning*) di `t5_tabel_eksperimen_per_fold.csv`.

**Himpunan fitur** (semua hanya memakai data sampai jam *t*, dihitung per stasiun):

| Himpunan | Isi |
|---|---|
| F1 | 29 fitur awal (04.1) |
| F2 | F1 ditambah 11 fitur cuaca dinamis (tren tekanan udara, selisih suhu dan titik embun, komponen angin u dan v, rata-rata kecepatan angin, jumlah jam angin lemah berturut-turut, hujan kumulatif 24 jam) dan 6 fitur statistik (maksimum, simpangan baku, dan rata-rata mingguan PM2.5, rasio PM2.5 dan PM10, rata-rata 24 jam CO dan NO2) |
| F3 | F2 ditambah 4 fitur **lintas stasiun**: rata-rata PM2.5 jam *t*, rata-rata PM2.5 24 jam, dan komponen angin u dan v dari **stasiun latih lain** pada jam yang sama (tanpa stasiun itu sendiri) |

**F3 mengubah aturan di bagian 01** ("tanpa mencampur deret antar stasiun"), jadi hasilnya dilaporkan terpisah dan memakainya adalah keputusan penulis. Di data uji, stasiun uji memakai rata-rata dari 9 stasiun latih pada periode uji. Itu sah secara waktu (hanya jam *t*), tetapi membuat uji "stasiun lain" tidak lagi sepenuhnya terpisah dari stasiun latih.

**Menjalankan ulang:** mengulang semua eksperimen memakan sekitar 45 menit pada 2 inti CPU. Karena itu `CARI_ULANG = False` (bawaan) membaca hasil yang sudah tersimpan di `Hasil/Tabel/`. Atur variabel lingkungan `CARI_ULANG=1` untuk menjalankan ulang semuanya.

In [ ]:
# False: baca hasil tersimpan (cepat). True atau variabel lingkungan CARI_ULANG=1: jalankan ulang semua eksperimen (sekitar 45 menit)
CARI_ULANG = os.environ.get("CARI_ULANG", "0") == "1"
BERKAS_EKS = os.path.join(TABEL, "t5_tabel_eksperimen_per_fold.csv")
print("Mode:", "jalankan ulang semua eksperimen" if CARI_ULANG else "baca hasil tersimpan di Hasil/Tabel")

#### Fitur tambahan, fitur lintas stasiun, dan pemeriksaan kebocorannya
Sel ini hanya berjalan bila `CARI_ULANG` aktif. Fitur lintas stasiun memakai rata-rata stasiun **lain** pada jam yang sama, jadi baris sebuah stasiun tidak memuat nilainya sendiri dua kali.

In [ ]:
if CARI_ULANG:
    from sklearn.linear_model import Ridge, ElasticNet
    from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
    from sklearn.model_selection import ParameterSampler

    # ---- Fitur tambahan. Semua memakai data sampai jam t (shift positif atau jendela ke belakang), dihitung per stasiun.
    def fitur_tambahan(data_stasiun):
        d = data_stasiun.set_index("waktu").sort_index()
        f = pd.DataFrame(index=d.index)
        for k in [3, 6, 24]:
            f[f"PRES_selisih{k}j"] = d["PRES"] - d["PRES"].shift(k)           # tren tekanan udara
        f["TEMP_selisih6j"] = d["TEMP"] - d["TEMP"].shift(6)
        f["TEMP_DEWP"] = d["TEMP"] - d["DEWP"]                                  # kelembapan relatif (kasar)
        sudut = np.radians(d["wd"].map(ARAH))
        f["angin_u"], f["angin_v"] = d["WSPM"] * np.sin(sudut), d["WSPM"] * np.cos(sudut)
        for w in [6, 24]:
            f[f"WSPM_rata{w}j"] = d["WSPM"].rolling(w, min_periods=w // 2).mean()
        lemah = (d["WSPM"] < 1.0).astype(int)                                   # angin lemah = udara tidak bergerak
        f["jam_angin_lemah"] = lemah.groupby((lemah == 0).cumsum()).cumsum().clip(upper=72)
        f["RAIN_jumlah24j"] = d["RAIN"].rolling(24, min_periods=12).sum()
        pm = d["PM2.5"]
        f["PM2.5_maks24j"] = pm.rolling(24, min_periods=12).max()
        f["PM2.5_sd24j"] = pm.rolling(24, min_periods=12).std()
        f["PM2.5_rata168j"] = pm.rolling(168, min_periods=84).mean()
        f["PM2.5_per_PM10"] = pm / d["PM10"].replace(0, np.nan)
        f["CO_rata24j"] = d["CO"].rolling(24, min_periods=12).mean()
        f["NO2_rata24j"] = d["NO2"].rolling(24, min_periods=12).mean()
        # kolom bantu untuk fitur lintas stasiun (bukan fitur)
        f["_pm_t"], f["_pm_rata24"] = pm, pm.rolling(24, min_periods=12).mean()
        return f

    KOL_CUACA = ["PRES_selisih3j", "PRES_selisih6j", "PRES_selisih24j", "TEMP_selisih6j", "TEMP_DEWP", "angin_u", "angin_v",
                 "WSPM_rata6j", "WSPM_rata24j", "jam_angin_lemah", "RAIN_jumlah24j"]
    KOL_STAT = ["PM2.5_maks24j", "PM2.5_sd24j", "PM2.5_rata168j", "PM2.5_per_PM10", "CO_rata24j", "NO2_rata24j"]
    KOL_LINTAS = ["lintas_pm_t", "lintas_pm_rata24j", "lintas_angin_u", "lintas_angin_v"]

    bagian = []
    for s in STASIUN_LATIH:
        f = fitur_tambahan(latih[latih["station"] == s]); f.insert(0, "stasiun", s); bagian.append(f)
    tambah = pd.concat(bagian)

    # Fitur lintas stasiun: rata-rata dari stasiun LAIN pada jam yang sama (tanpa stasiun itu sendiri), hanya jam t
    for nama, kol in [("lintas_pm_t", "_pm_t"), ("lintas_pm_rata24j", "_pm_rata24"), ("lintas_angin_u", "angin_u"), ("lintas_angin_v", "angin_v")]:
        lebar = tambah.reset_index().pivot(index="waktu", columns="stasiun", values=kol)
        jumlah, banyak = lebar.sum(axis=1), lebar.notna().sum(axis=1)
        lain = {s: (jumlah - lebar[s].fillna(0)) / (banyak - lebar[s].notna().astype(int)).replace(0, np.nan) for s in lebar.columns}
        tambah[nama] = pd.concat([lain[s].rename("v").to_frame().assign(stasiun=s).set_index("stasiun", append=True)["v"]
                                  for s in lebar.columns]).reindex(pd.MultiIndex.from_arrays([tambah.index, tambah["stasiun"]])).to_numpy()

    kunci = pd.MultiIndex.from_arrays([fitur_latih.index, fitur_latih["stasiun"].to_numpy()])
    tmb = tambah.set_index("stasiun", append=True)
    assert not tmb.index.duplicated().any()
    tmb = tmb.reindex(kunci)
    XA = pd.concat([fitur_latih[KOLOM_FITUR].reset_index(drop=True),
                    tmb[KOL_CUACA + KOL_STAT + KOL_LINTAS].reset_index(drop=True)], axis=1)
    XA.index = fitur_latih.index
    YA = fitur_latih["target"].to_numpy()
    HIMPUNAN = {"F1 (29 fitur awal)": KOLOM_FITUR,
                "F1 + cuaca dinamis": KOLOM_FITUR + KOL_CUACA,
                "F1 + statistik PM dan polutan": KOLOM_FITUR + KOL_STAT,
                "F2 (F1 + cuaca + statistik)": KOLOM_FITUR + KOL_CUACA + KOL_STAT,
                "F3 (F2 + lintas stasiun)": KOLOM_FITUR + KOL_CUACA + KOL_STAT + KOL_LINTAS}
    print({k: len(v) for k, v in HIMPUNAN.items()})
    print("Kosong per kelompok fitur baru (%):", XA[KOL_CUACA + KOL_STAT + KOL_LINTAS].isna().mean().mul(100).round(1).to_dict())

    # Indeks baris latih dan validasi tiap fold. Baris validasi = baris yang dipakai naive (PM2.5 jam t tidak kosong).
    FOLD = []
    for jam_a, jam_b in fold_cv:
        assert jam_a.max() + pd.Timedelta(hours=HORIZON) < jam_b.min()
        a, b = XA.index.isin(jam_a), XA.index.isin(jam_b)
        ada = XA["PM2.5"].to_numpy()[b] == XA["PM2.5"].to_numpy()[b]
        FOLD.append((np.flatnonzero(a), np.flatnonzero(b)[ada]))
    for k, (ia, ib) in enumerate(FOLD):
        assert abs(rmse(YA[ib], XA["PM2.5"].to_numpy()[ib]) - skor["Naive forecast"]["RMSE"][k]) < 1e-9
    print("Baris validasi dan galat naive sama dengan 04.5:", [len(ib) for _, ib in FOLD])

#### Langkah 1: ablasi fitur
Pemeriksaan kebocoran fitur tambahan (identik dengan atau tanpa data sesudah *T0*) dan ablasi dengan model linear.

In [ ]:
if CARI_ULANG:
    import time
    # Uji kebocoran fitur tambahan: nilai sampai T0 harus sama dengan atau tanpa data sesudah T0
    T0 = pd.Timestamp("2015-06-01 00:00")
    for s in (STASIUN_LATIH[0], STASIUN_LATIH[4]):
        d_s = latih[latih["station"] == s]
        penuh = fitur_tambahan(d_s).loc[:T0]
        potong = fitur_tambahan(d_s[d_s["waktu"] <= T0]).loc[:T0]
        pd.testing.assert_frame_equal(penuh, potong)
    print("Fitur tambahan tidak melihat masa depan (identik dengan atau tanpa data sesudah T0).")

    # ---- Pembantu: praproses per fold (imputer median dan scaler dilatih hanya pada data latih fold), model, dan pencatatan
    HASIL = []     # satu baris per (eksperimen, fold)
    def catat(kelompok, nama, k, yb, tebak, **extra):
        HASIL.append({"kelompok": kelompok, "eksperimen": nama, "fold": k + 1, "RMSE": rmse(yb, tebak), "MAE": mae(yb, tebak), **extra})

    _cache_prep = {}
    def siap(himpunan, k):
        kunci_c = (himpunan, k)
        if kunci_c not in _cache_prep:
            ia, ib = FOLD[k]
            kol = HIMPUNAN[himpunan]
            pra = Pipeline([("isi_kosong", SimpleImputer(strategy="median")), ("skala", StandardScaler())])
            Xa = pra.fit_transform(XA.iloc[ia][kol].to_numpy())
            Xb = pra.transform(XA.iloc[ib][kol].to_numpy())
            _cache_prep[kunci_c] = (Xa, YA[ia], Xb, YA[ib])
        return _cache_prep[kunci_c]

    def hgb(**p):
        base = dict(max_iter=200, learning_rate=0.1, max_depth=4, early_stopping=False, random_state=SEED)
        base.update(p)
        return HistGradientBoostingRegressor(**base)

    # ---- Langkah 1: ablasi fitur dengan Ridge dan Elastic Net (murah)
    t0 = time.time()
    for hs in HIMPUNAN:
        for k in range(len(FOLD)):
            Xa, ya, Xb, yb = siap(hs, k)
            catat("1 ablasi fitur", f"Ridge | {hs}", k, yb, Ridge(alpha=1.0).fit(Xa, ya).predict(Xb))
            catat("1 ablasi fitur", f"Elastic Net (alpha=0,1) | {hs}", k, yb,
                  ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000, random_state=SEED).fit(Xa, ya).predict(Xb))
    print(f"Langkah 1 selesai ({time.time()-t0:.0f} dtk)")
    tampil = pd.DataFrame(HASIL).groupby("eksperimen")["RMSE"].agg(["mean", "std"]).round(2)
    print(tampil)

#### Langkah 2: keluarga model pada F1, F2, dan F3

In [ ]:
if CARI_ULANG:
    # ---- Langkah 2: keluarga model pada F1, F2, dan F3 (HGB dengan pengaturan 04.6: 200 iterasi, lr 0,1, kedalaman 4)
    t0 = time.time()
    PRED_CACHE = {}
    for hs in ["F1 (29 fitur awal)", "F2 (F1 + cuaca + statistik)", "F3 (F2 + lintas stasiun)"]:
        for k in range(len(FOLD)):
            Xa, ya, Xb, yb = siap(hs, k)
            rg = Ridge(alpha=1.0).fit(Xa, ya)
            p_rg = rg.predict(Xb)
            p_hgb = hgb().fit(Xa, ya).predict(Xb)
            sisa = hgb().fit(Xa, ya - rg.predict(Xa)).predict(Xb)          # boosting pada residual Ridge (dilatih di data latih fold)
            catat("2 model", f"Gradient boosting | {hs}", k, yb, p_hgb)
            catat("2 model", f"Ridge lalu boosting pada residual | {hs}", k, yb, p_rg + sisa)
            catat("2 model", f"Rata-rata Ridge dan boosting (0,5) | {hs}", k, yb, 0.5 * (p_rg + p_hgb))
            if hs != "F1 (29 fitur awal)":
                rf = RandomForestRegressor(n_estimators=100, min_samples_leaf=30, max_features=0.4, max_samples=0.25,
                                           n_jobs=2, random_state=SEED).fit(Xa, ya)
                catat("2 model", f"Random forest | {hs}", k, yb, rf.predict(Xb))
        print(f"  {hs} selesai ({time.time()-t0:.0f} dtk)", flush=True)
    r2 = pd.DataFrame(HASIL); r2 = r2[r2["kelompok"] == "2 model"]
    print(r2.groupby("eksperimen")["RMSE"].agg(["mean", "std"]).round(2).to_string())

#### Langkah 3: *tuning* acak
Jumlah iterasi *boosting* dipilih dari kurva RMSE rata-rata 5 *fold* (satu nilai untuk semua *fold*), seperti hiperparameter lain. Konfigurasi terbaik dipilih dari rata-rata RMSE validasi, sehingga skornya sedikit optimistis.

In [ ]:
if CARI_ULANG:
    # ---- Langkah 3: tuning acak HGB dan Random forest pada F2 dan F3 (anggaran tertulis: 12 konfigurasi HGB dan 6 konfigurasi RF per himpunan fitur)
    TUNING = []
    RUANG_HGB = {"learning_rate": [0.03, 0.05, 0.1], "max_depth": [3, 4, 6, 8], "min_samples_leaf": [20, 100, 300, 1000],
                 "l2_regularization": [0, 1, 10, 100], "max_leaf_nodes": [15, 31, 63], "max_features": [0.5, 0.8, 1.0]}
    RUANG_RF = {"min_samples_leaf": [10, 30, 100], "max_features": [0.3, 0.5], "max_samples": [0.25, 0.5]}
    UNDI_HGB = list(ParameterSampler(RUANG_HGB, n_iter=12, random_state=SEED))
    UNDI_RF = list(ParameterSampler(RUANG_RF, n_iter=6, random_state=SEED))
    TITIK = 20
    TERPILIH = {}

    t0 = time.time()
    for hs in ["F2 (F1 + cuaca + statistik)", "F3 (F2 + lintas stasiun)"]:
        # HGB: satu latihan per konfigurasi per fold, RMSE dihitung di tiap 20 iterasi lewat staged_predict.
        # Jumlah iterasi dipilih dari rata-rata kurva 5 fold (satu nilai untuk semua fold), sama seperti hiperparameter lain.
        for n, p in enumerate(UNDI_HGB):
            mi = 600 if p["learning_rate"] < 0.1 else 300
            kurva = []
            for k in range(len(FOLD)):
                Xa, ya, Xb, yb = siap(hs, k)
                m = hgb(max_iter=mi, **p).fit(Xa, ya)
                kurva.append([rmse(yb, pr) for i, pr in enumerate(m.staged_predict(Xb), 1) if i % TITIK == 0])
            kurva = np.array(kurva); rata = kurva.mean(axis=0); j = int(rata.argmin())
            TUNING.append({"model": "HGB", "himpunan": hs, "konfigurasi": n, **p, "iterasi_terbaik": (j + 1) * TITIK,
                           "RMSE rata-rata": rata[j], "RMSE sd": kurva[:, j].std(ddof=1)})
            print(f"  HGB {hs[:2]} #{n} {p} -> iterasi {(j+1)*TITIK}, RMSE {rata[j]:.2f} ({time.time()-t0:.0f} dtk)", flush=True)
        th = pd.DataFrame([t for t in TUNING if t["model"] == "HGB" and t["himpunan"] == hs])
        b = th.loc[th["RMSE rata-rata"].idxmin()]
        TERPILIH[("HGB", hs)] = {c: b[c] for c in RUANG_HGB} | {"max_iter": int(b["iterasi_terbaik"])}
        # Random forest
        for n, p in enumerate(UNDI_RF):
            r = []
            for k in range(len(FOLD)):
                Xa, ya, Xb, yb = siap(hs, k)
                rf = RandomForestRegressor(n_estimators=100, n_jobs=2, random_state=SEED, **p).fit(Xa, ya)
                r.append(rmse(yb, rf.predict(Xb)))
            TUNING.append({"model": "RF", "himpunan": hs, "konfigurasi": n, **p, "RMSE rata-rata": np.mean(r), "RMSE sd": np.std(r, ddof=1)})
            print(f"  RF {hs[:2]} #{n} {p} -> RMSE {np.mean(r):.2f} ({time.time()-t0:.0f} dtk)", flush=True)
        tr = pd.DataFrame([t for t in TUNING if t["model"] == "RF" and t["himpunan"] == hs])
        b = tr.loc[tr["RMSE rata-rata"].idxmin()]
        TERPILIH[("RF", hs)] = {c: b[c] for c in RUANG_RF}
    tabel_tuning = pd.DataFrame(TUNING)
    tabel_tuning.round(3).to_csv(os.path.join(TABEL, "t5_tabel_tuning.csv"), index=False)
    print(TERPILIH)

#### Langkah 4 dan 5: rata-rata model dan transformasi target
Rata-rata memakai bobot sama dan tidak disetel. Transformasi target dikembalikan ke skala asli tanpa koreksi bias (RMSE dihitung di skala asli).

In [ ]:
if CARI_ULANG:
    # ---- Langkah 4 dan 5: konfigurasi terpilih dilatih ulang (5 fold), lalu rata-rata beberapa model dan transformasi target
    PRED = {}
    def simpan(nama, hs, k, yb, tebak, kel):
        PRED[(nama, hs, k)] = tebak
        catat(kel, f"{nama} | {hs}", k, yb, tebak)

    t0 = time.time()
    for hs in ["F2 (F1 + cuaca + statistik)", "F3 (F2 + lintas stasiun)"]:
        ph = {c: (int(v) if c in ("max_depth", "min_samples_leaf", "max_leaf_nodes", "max_iter") else float(v)) for c, v in TERPILIH[("HGB", hs)].items()}
        pr = {c: (int(v) if c == "min_samples_leaf" else float(v)) for c, v in TERPILIH[("RF", hs)].items()}
        for k in range(len(FOLD)):
            Xa, ya, Xb, yb = siap(hs, k)
            rg = Ridge(alpha=1.0).fit(Xa, ya)
            en = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000, random_state=SEED).fit(Xa, ya)
            PRED[("Ridge", hs, k)] = rg.predict(Xb); PRED[("EN", hs, k)] = en.predict(Xb)
            simpan("HGB terpilih", hs, k, yb, hgb(**ph).fit(Xa, ya).predict(Xb), "3 tuning")
            simpan("Random forest terpilih", hs, k, yb,
                   RandomForestRegressor(n_estimators=100, n_jobs=2, random_state=SEED, **pr).fit(Xa, ya).predict(Xb), "3 tuning")
            P = lambda n: PRED[(n, hs, k)]
            catat("4 rata-rata", f"Ridge + HGB terpilih | {hs}", k, yb, (P("Ridge") + P("HGB terpilih")) / 2)
            catat("4 rata-rata", f"Ridge + Random forest terpilih | {hs}", k, yb, (P("Ridge") + P("Random forest terpilih")) / 2)
            catat("4 rata-rata", f"Ridge + HGB + Random forest terpilih | {hs}", k, yb, (P("Ridge") + P("HGB terpilih") + P("Random forest terpilih")) / 3)
            catat("4 rata-rata", f"Elastic Net + HGB + Random forest terpilih | {hs}", k, yb, (P("EN") + P("HGB terpilih") + P("Random forest terpilih")) / 3)
            # transformasi target: model dilatih pada skala baru, prediksi dikembalikan ke skala asli (tanpa koreksi bias, dipotong di 0)
            for tn, f_, g_ in [("akar kuadrat", np.sqrt, np.square), ("log1p", np.log1p, np.expm1)]:
                yt = f_(ya)
                catat("5 transformasi target", f"Ridge, target {tn} | {hs}", k, yb, g_(np.clip(Ridge(alpha=1.0).fit(Xa, yt).predict(Xb), 0, None)))
                catat("5 transformasi target", f"HGB terpilih, target {tn} | {hs}", k, yb, g_(np.clip(hgb(**ph).fit(Xa, yt).predict(Xb), 0, None)))
        print(f"  {hs[:2]} selesai ({time.time()-t0:.0f} dtk)", flush=True)

#### Langkah 6: putaran 2

In [ ]:
if CARI_ULANG:
    # ---- Langkah 6 (putaran 2, ditentukan SETELAH melihat langkah 3): konfigurasi Random forest terbaik ada di tepi grid
    # (min_samples_leaf terbesar, max_features dan max_samples terkecil), jadi grid diperluas ke arah regularisasi lebih kuat.
    # Hanya F3. Ditambah Extra Trees dengan regularisasi serupa. Anggaran: 6 konfigurasi RF dan 3 konfigurasi Extra Trees.
    from sklearn.ensemble import ExtraTreesRegressor
    hs3 = "F3 (F2 + lintas stasiun)"
    RF2 = [dict(min_samples_leaf=100, max_features=0.2, max_samples=0.25), dict(min_samples_leaf=300, max_features=0.3, max_samples=0.25),
           dict(min_samples_leaf=300, max_features=0.2, max_samples=0.25), dict(min_samples_leaf=1000, max_features=0.3, max_samples=0.25),
           dict(min_samples_leaf=300, max_features=0.3, max_samples=0.1), dict(min_samples_leaf=1000, max_features=0.2, max_samples=0.1)]
    ET2 = [dict(min_samples_leaf=50, max_features=0.3), dict(min_samples_leaf=100, max_features=0.3), dict(min_samples_leaf=300, max_features=0.3)]
    PRED2, SKOR2 = {}, []
    t0 = time.time()
    for jenis, daftar in [("RF", RF2), ("ET", ET2)]:
        for n, p in enumerate(daftar):
            r = []
            for k in range(len(FOLD)):
                Xa, ya, Xb, yb = siap(hs3, k)
                if jenis == "RF":
                    m = RandomForestRegressor(n_estimators=100, n_jobs=2, random_state=SEED, **p)
                else:
                    m = ExtraTreesRegressor(n_estimators=100, n_jobs=2, random_state=SEED, bootstrap=True, max_samples=0.25, **p)
                tebak = m.fit(Xa, ya).predict(Xb)
                PRED2[(jenis, n, k)] = tebak
                catat("6 putaran 2", f"{'Random forest' if jenis == 'RF' else 'Extra Trees'} {p} | {hs3}", k, yb, tebak)
                r.append(rmse(yb, tebak))
            SKOR2.append({"model": jenis, "konfigurasi": n, **p, "RMSE rata-rata": np.mean(r), "RMSE sd": np.std(r, ddof=1)})
            print(f"  {jenis} #{n} {p} -> RMSE {np.mean(r):.2f} ({time.time()-t0:.0f} dtk)", flush=True)
    tabel_tuning2 = pd.DataFrame(SKOR2)
    tabel_tuning2.round(3).to_csv(os.path.join(TABEL, "t5_tabel_tuning_putaran2.csv"), index=False)
    n_rf = int(tabel_tuning2[tabel_tuning2["model"] == "RF"]["RMSE rata-rata"].idxmin())
    n_et = int(tabel_tuning2[tabel_tuning2["model"] == "ET"]["RMSE rata-rata"].idxmin()) - len(RF2)
    for k in range(len(FOLD)):
        Xa, ya, Xb, yb = siap(hs3, k)
        p_rg = Ridge(alpha=1.0).fit(Xa, ya).predict(Xb)
        catat("6 putaran 2", f"Ridge + Random forest putaran 2 terbaik | {hs3}", k, yb, (p_rg + PRED2[("RF", n_rf, k)]) / 2)
        catat("6 putaran 2", f"Ridge + Extra Trees terbaik | {hs3}", k, yb, (p_rg + PRED2[("ET", n_et, k)]) / 2)
        catat("6 putaran 2", f"Ridge + Random forest + Extra Trees terbaik | {hs3}", k, yb, (p_rg + PRED2[("RF", n_rf, k)] + PRED2[("ET", n_et, k)]) / 3)
    hasil_eks = pd.DataFrame(HASIL)
    hasil_eks.round(4).to_csv(BERKAS_EKS, index=False)
    print(hasil_eks[hasil_eks["kelompok"] == "6 putaran 2"].groupby("eksperimen")["RMSE"].agg(["mean", "std"]).round(2).to_string())

#### Ringkasan semua eksperimen
Tabel diurutkan dari RMSE terendah. Selisih terhadap Ridge F1 dihitung berpasangan per *fold* (Ridge pada 29 fitur awal, hasil 05.1).

In [ ]:
if not CARI_ULANG:
    hasil_eks = pd.read_csv(BERKAS_EKS)
    tuning_semua = pd.concat([pd.read_csv(os.path.join(TABEL, "t5_tabel_tuning.csv")),
                              pd.read_csv(os.path.join(TABEL, "t5_tabel_tuning_putaran2.csv"))], ignore_index=True)
else:
    tuning_semua = pd.concat([tabel_tuning, tabel_tuning2], ignore_index=True)

nv = rmse_naive.mean()
rid = hasil_eks[hasil_eks["eksperimen"] == "Ridge | F1 (29 fitur awal)"].set_index("fold")["RMSE"]
baris = []
for nama_e, g in hasil_eks.groupby("eksperimen"):
    g = g.set_index("fold"); d = g["RMSE"] - rid
    baris.append({"Eksperimen": nama_e, "Langkah": g["kelompok"].iloc[0], "RMSE rata-rata": g["RMSE"].mean(),
                  "RMSE sd": g["RMSE"].std(ddof=1), "MAE rata-rata": g["MAE"].mean(),
                  "Selisih vs Ridge F1 (rata-rata)": d.mean(), "sd selisih": d.std(ddof=1),
                  "Fold lebih baik dari Ridge F1": f"{int((d < 0).sum())} dari {len(d)}",
                  "Perbaikan vs naive (%)": 100 * (nv - g["RMSE"].mean()) / nv})
ringkas = pd.DataFrame(baris).set_index("Eksperimen").sort_values("RMSE rata-rata")
ringkas.round(2).to_csv(os.path.join(TABEL, "t5_tabel_eksperimen_ringkas.csv"))
print(f"Eksperimen berbeda: {len(ringkas)} | konfigurasi tuning: {len(tuning_semua)} | rata-rata dan sd semua konfigurasi tuning di t5_tabel_tuning*.csv, nilai tiap fold di t5_tabel_eksperimen_per_fold.csv")
print("\nDelapan terbaik:")
print(ringkas.head(8).round(2).drop(columns="Langkah").to_string())
tanpa_lintas = ringkas[~ringkas.index.str.contains("F3")]
print("\nTerbaik tanpa fitur lintas stasiun (F1 atau F2):", tanpa_lintas.index[0], f"| RMSE {tanpa_lintas.iloc[0]['RMSE rata-rata']:.2f}")
terbaik_e = ringkas.index[0]
print(f"RMSE terendah dari semua eksperimen: {ringkas.iloc[0]['RMSE rata-rata']:.2f} ({terbaik_e}). Di bawah 70: {ringkas.iloc[0]['RMSE rata-rata'] < 70}")

# Dua cara membaca hasil terbaik (aturan ditulis sebelum hasil dilihat)
b = ringkas.iloc[0]
cara1 = b["RMSE rata-rata"] - ringkas.loc["Ridge | F1 (29 fitur awal)", "RMSE rata-rata"] < -ringkas.iloc[0]["RMSE sd"]
cara2 = (int(b["Fold lebih baik dari Ridge F1"].split()[0]) >= 4 and -b["Selisih vs Ridge F1 (rata-rata)"] > 3 * b["sd selisih"]
         and -b["Selisih vs Ridge F1 (rata-rata)"] > 0.01 * ringkas.loc["Ridge | F1 (29 fitur awal)", "RMSE rata-rata"])
print("\nAturan protokol butir 2 (selisih harus melebihi satu simpangan baku antar fold):", "terbedakan dari Ridge F1" if cara1 else "TIDAK terbedakan dari Ridge F1, pilih yang sederhana")
print("Aturan berpasangan (lebih baik di minimal 4 dari 5 fold, rata-rata selisih lebih dari 3 sd selisih dan lebih dari 1% RMSE):",
      "terpenuhi" if cara2 else "tidak terpenuhi")
print("Kedua aturan memberi jawaban berbeda bila hasilnya tidak sama. Memilih aturan mana yang dipakai adalah keputusan penulis.")

# Konfigurasi terbaik tiap model: apakah ada di tepi grid yang dicoba?
print()
for (mdl, hs), g in tuning_semua.groupby(["model", "himpunan"]):
    best = g.loc[g["RMSE rata-rata"].idxmin()]
    tepi = []
    for c in ["min_samples_leaf", "max_features", "max_samples", "max_depth", "learning_rate", "l2_regularization", "max_leaf_nodes"]:
        if c in g and g[c].notna().any() and g[c].nunique() > 1 and best[c] in (g[c].min(), g[c].max()):
            tepi.append(c)
    print(f"{mdl} pada {hs[:2]}: terbaik RMSE {best['RMSE rata-rata']:.2f}, parameter di tepi grid: {tepi if tepi else 'tidak ada'}")

# Grafik: naive, Ridge F1, terbaik tanpa lintas stasiun, terbaik dengan lintas stasiun, dengan garis 70
pilihan = [("Naive forecast", None), ("Ridge F1 (05.1)", "Ridge | F1 (29 fitur awal)"),
           ("Terbaik tanpa lintas stasiun", tanpa_lintas.index[0]), ("Terbaik dengan lintas stasiun", ringkas[ringkas.index.str.contains("F3")].index[0])]
fig, ax = plt.subplots(figsize=(9, 3.8))
for i, (lab, kunci_e) in enumerate(pilihan[::-1]):
    nilai = rmse_naive if kunci_e is None else hasil_eks[hasil_eks["eksperimen"] == kunci_e].sort_values("fold")["RMSE"].to_numpy()
    warna = ABU if kunci_e is None else (BIRU if "Ridge |" in kunci_e else ORANYE)
    ax.barh(i, nilai.mean(), xerr=nilai.std(ddof=1), color=warna, height=0.55, alpha=0.9, error_kw={"ecolor": "#2c2c2a", "capsize": 3, "lw": 1})
    ax.scatter(nilai, [i] * len(nilai), s=14, color="#2c2c2a", zorder=3)
    ax.text(0.8, i, f"{nilai.mean():.1f}", va="center", ha="left", color="white", fontsize=9, fontweight="bold")
ax.axvline(70, color=ORANYE, ls="--", lw=1.2)
ax.set_yticks(range(len(pilihan))); ax.set_yticklabels([p_[0] for p_ in pilihan[::-1]])
judul(ax, "RMSE validasi silang: eksperimen lanjutan", "Batang = rata-rata 5 fold, garis = ± sd, titik = tiap fold, putus-putus = 70")
ax.set(xlabel="RMSE (µg/m³), makin kecil makin baik"); ax.grid(axis="y", visible=False)
simpan_gambar("t5_02_eksperimen_lanjutan.png")

### 05.6 Cara membaca hasil 05.5 dan keterbatasannya
* **Banyak konfigurasi pada validasi yang sama.** Skor terbaik dari puluhan konfigurasi pasti lebih optimistis daripada skor model yang dipilih sebelum melihat hasil. Angkanya tidak boleh dilaporkan sebagai kinerja akhir. Kinerja akhir hanya dari data uji, satu kali.
* **Dua aturan pemilihan bisa berbeda jawaban.** Aturan protokol butir 2 (selisih harus melebihi satu simpangan baku antar *fold*, sekitar 19) hampir tidak pernah terpenuhi karena simpangan baku itu didominasi beda kesulitan antar periode. Aturan berpasangan membandingkan *fold* yang sama sehingga lebih tajam, tetapi tetap hanya lima *fold* dan konfigurasi yang dicoba banyak. Ambangnya (minimal 4 dari 5 *fold*, selisih rata-rata lebih dari 3 kali simpangan baku selisih dan lebih dari 1% RMSE) adalah usulan Claude yang ditetapkan setelah melihat sebagian hasil, jadi statusnya [Usulan] dan bukan aturan yang dikunci sebelumnya. Memilih aturan adalah keputusan penulis.
* **F3 (lintas stasiun) mengubah desain.** Bila dipakai, aturan di bagian 01 harus diubah, dan perlu diputuskan apakah uji di stasiun uji masih dianggap pergeseran distribusi yang murni.
* **Konfigurasi terbaik sering ada di tepi grid** (lihat keluaran di atas). Grid diperluas sekali (langkah 6). Perluasan lebih jauh bisa menambah sedikit, tetapi hasil putaran 2 menunjukkan perbaikannya sudah mendatar.
* **Penjelasan mengapa fitur lintas stasiun dan *random forest* membantu belum diuji.** Dugaan: menyaring *noise* dan menangkap kondisi regional. Itu hipotesis.
* **Target `log1p` dan akar kuadrat dikembalikan ke skala asli tanpa koreksi bias**, jadi hasilnya bisa lebih buruk daripada versi yang dikoreksi. Hanya dicoba tanpa koreksi.
* **Fitur cuaca dan statistik tambahan** diuji dengan ablasi di langkah 1. Dampaknya pada model linear dibaca dari tabel (kecil), dan pada model pohon dibaca dari langkah 2 dan 3.

### 05.7 *Tuning* sistematis model linear (kandidat utama)
Dua kandidat utama adalah Ridge dan Elastic Net (hasil 05.1: selisih RMSE 0,41, jauh di bawah simpangan baku antar *fold* sekitar 19). Grid di bawah **ditetapkan sebelum dijalankan** (22 Nov 2026, [Usulan], disetujui penulis dengan meminta *tuning* dijalankan) dan tidak diubah setelah melihat hasil:

| Model | Parameter | Grid | Konfigurasi |
|---|---|---|---|
| Ridge | `alpha` | 0,1; 1; 10; 100; 1000 | 5 |
| Elastic Net | `alpha` dan `l1_ratio` | `alpha` 0,01; 0,1; 1; 3; 10 dan `l1_ratio` 0,1; 0,5; 0,9 | 15 |

Dikunci: 29 fitur awal (F1), imputer median, *scaler* standar, `fit_intercept=True`, `max_iter=5000`. Aturan pemilihan (ditulis sebelum hasil dilihat):

1. Dasar: rata-rata RMSE validasi silang 5 *fold*, *fold* dan baris validasi sama dengan 05.1 (`assert` galat *naive*).
2. Keluarga model dipilih dengan aturan protokol butir 2: bila selisih rata-rata RMSE dengan konfigurasi terbaik lebih kecil dari satu simpangan baku antar *fold*, konfigurasi itu tidak terbedakan, dan **Ridge lebih sederhana daripada Elastic Net**.
3. Di dalam keluarga terpilih, ambil konfigurasi dengan rata-rata RMSE terendah. Bila ada di tepi grid, dilaporkan sebagai keterbatasan dan grid tidak diperluas lagi.
4. Model final harus lebih baik dari *naive forecast*. **Data uji tidak dibuka.** Hasilnya kandidat, bukan model final, sampai konfigurasi dibekukan dan di-*commit*.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

GRID_RIDGE = [0.1, 1, 10, 100, 1000]
GRID_EN_ALPHA = [0.01, 0.1, 1, 3, 10]
GRID_EN_L1 = [0.1, 0.5, 0.9]

X7, y7 = fitur_latih[KOLOM_FITUR], fitur_latih["target"]
fold7 = []                       # praproses dilatih per fold hanya pada data latih fold itu, lalu dipakai ulang untuk semua konfigurasi
for k, (jam_a, jam_b) in enumerate(fold_cv):
    assert jam_a.max() + pd.Timedelta(hours=HORIZON) < jam_b.min()
    a, b = X7.index.isin(jam_a), X7.index.isin(jam_b)
    ada = X7[b]["PM2.5"].notna().to_numpy()
    pra = Pipeline([("isi_kosong", SimpleImputer(strategy="median")), ("skala", StandardScaler())])
    Xa, ya = pra.fit_transform(X7[a]), y7[a].to_numpy()
    Xv, yv = pra.transform(X7[b][ada]), y7[b][ada].to_numpy()
    assert abs(rmse(yv, X7[b][ada]["PM2.5"].to_numpy()) - skor["Naive forecast"]["RMSE"][k]) < 1e-9
    fold7.append((Xa, ya, Xv, yv))

KONFIG7 = [(f"Ridge (alpha={al})", "Ridge", al, np.nan, (lambda al=al: Ridge(alpha=al))) for al in GRID_RIDGE]
KONFIG7 += [(f"Elastic Net (alpha={al}, l1_ratio={l1})", "Elastic Net", al, l1,
             (lambda al=al, l1=l1: ElasticNet(alpha=al, l1_ratio=l1, max_iter=5000, random_state=SEED)))
            for al in GRID_EN_ALPHA for l1 in GRID_EN_L1]

per_fold7, peringatan7 = [], {}
for nama, kel, al, l1, buat in KONFIG7:
    n_peringatan = 0
    for k, (Xa, ya, Xv, yv) in enumerate(fold7):
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always", ConvergenceWarning)
            tebak = buat().fit(Xa, ya).predict(Xv)
        n_peringatan += sum(issubclass(x.category, ConvergenceWarning) for x in w)
        per_fold7.append({"model": nama, "keluarga": kel, "alpha": al, "l1_ratio": l1, "fold": k + 1,
                          "RMSE": rmse(yv, tebak), "MAE": mae(yv, tebak)})
    peringatan7[nama] = n_peringatan
pf7 = pd.DataFrame(per_fold7)
pf7.round(4).to_csv(os.path.join(TABEL, "t5_tabel_tuning_linear_per_fold.csv"), index=False)

acuan = pf7[pf7["model"] == "Ridge (alpha=1)"].sort_values("fold")["RMSE"].to_numpy()   # kandidat sebelumnya, untuk selisih berpasangan
baris7 = []
for nama, kel, al, l1, _ in KONFIG7:
    g = pf7[pf7["model"] == nama].sort_values("fold"); d = g["RMSE"].to_numpy() - acuan
    baris7.append({"Model": nama, "Keluarga": kel, "alpha": al, "l1_ratio": l1,
                   "RMSE rata-rata": g["RMSE"].mean(), "RMSE sd": g["RMSE"].std(ddof=1),
                   "MAE rata-rata": g["MAE"].mean(), "MAE sd": g["MAE"].std(ddof=1),
                   "Selisih vs Ridge alpha=1": d.mean(), "sd selisih": d.std(ddof=1),
                   "Fold lebih baik dari Ridge alpha=1": f"{int((d < 0).sum())} dari {len(d)}",
                   "Peringatan konvergensi": peringatan7[nama]})
ringkas7 = pd.DataFrame(baris7).set_index("Model")
ringkas7.round(3).to_csv(os.path.join(TABEL, "t5_tabel_tuning_linear.csv"))
print("Total konfigurasi:", len(ringkas7), "| total peringatan konvergensi:", int(ringkas7["Peringatan konvergensi"].sum()))
print(ringkas7.sort_values("RMSE rata-rata").round(2).drop(columns=["Keluarga"]).to_string())

# Aturan pemilihan (ditulis sebelum hasil dilihat, lihat markdown di atas)
terbaik7 = ringkas7["RMSE rata-rata"].idxmin()
sd7 = ringkas7.loc[terbaik7, "RMSE sd"]
setara7 = ringkas7[ringkas7["RMSE rata-rata"] - ringkas7.loc[terbaik7, "RMSE rata-rata"] < sd7]
keluarga7 = "Ridge" if (setara7["Keluarga"] == "Ridge").any() else "Elastic Net"
pilih7 = ringkas7[ringkas7["Keluarga"] == keluarga7]["RMSE rata-rata"].idxmin()
grid_k = GRID_RIDGE if keluarga7 == "Ridge" else GRID_EN_ALPHA
tepi7 = ringkas7.loc[pilih7, "alpha"] in (min(grid_k), max(grid_k))
print(f"\nRMSE terendah          : {terbaik7} ({ringkas7.loc[terbaik7, 'RMSE rata-rata']:.2f}), sd antar fold {sd7:.2f}")
print(f"Tidak terbedakan       : {len(setara7)} dari {len(ringkas7)} konfigurasi")
print(f"Keluarga menurut aturan: {keluarga7}")
print(f"Kandidat terpilih      : {pilih7} | RMSE {ringkas7.loc[pilih7, 'RMSE rata-rata']:.2f} ± {ringkas7.loc[pilih7, 'RMSE sd']:.2f}, "
      f"MAE {ringkas7.loc[pilih7, 'MAE rata-rata']:.2f} ± {ringkas7.loc[pilih7, 'MAE sd']:.2f}")
print(f"Lebih baik dari naive ({rmse_naive.mean():.2f}): {bool(ringkas7.loc[pilih7, 'RMSE rata-rata'] < rmse_naive.mean())} | alpha di tepi grid: {bool(tepi7)}")
print("Ini kandidat, belum model final: konfigurasi belum dibekukan dan data uji belum dibuka.")

# Grafik: RMSE rata-rata terhadap alpha
fig, ax = plt.subplots(figsize=(9, 3.8))
r = ringkas7[ringkas7["Keluarga"] == "Ridge"]
ax.plot(r["alpha"], r["RMSE rata-rata"], marker="o", color=BIRU, label="Ridge")
for l1, warna in zip(GRID_EN_L1, [ORANYE, "#a14a1f", ABU]):
    e = ringkas7[(ringkas7["Keluarga"] == "Elastic Net") & (ringkas7["l1_ratio"] == l1)]
    ax.plot(e["alpha"], e["RMSE rata-rata"], marker="s", color=warna, label=f"Elastic Net, l1_ratio {l1}")
ax.axhline(rmse_naive.mean(), color=ABU, ls="--", lw=1)
ax.set_xscale("log")
judul(ax, "Tuning model linear: RMSE validasi silang", "Rata-rata 5 fold, putus-putus = naive forecast (selisih antar titik jauh di bawah sd antar fold, sekitar 19)")
ax.set(xlabel="alpha (skala log)", ylabel="RMSE (µg/m³)"); ax.legend(frameon=False, fontsize=8)
simpan_gambar("t5_03_tuning_linear.png")

### 05.8 Cara membaca hasil 05.7
* Selisih RMSE antar konfigurasi linear sangat kecil dibanding simpangan baku antar *fold* (sekitar 19), jadi tidak ada konfigurasi yang boleh disebut nyata lebih baik. Yang berarti adalah bentuk kurva: apakah RMSE datar di sekitar nilai terpilih atau naik tajam di salah satu sisi.
* Pemilihan dari 20 konfigurasi pada validasi yang sama tetap sedikit optimistis. Angka validasi bukan kinerja akhir.
* Tidak ada fitur, praproses, atau *fold* yang diubah di bagian ini, sehingga selisihnya murni dari hiperparameter.
* Grid tidak diperluas setelah melihat hasil. Bila optimum ada di tepi grid, itu dilaporkan, bukan dikejar.

## Ringkasan
Angka di bawah dihitung langsung dari data latih di atas.

In [ ]:
n_uji = fitur_uji["kelompok_uji"].value_counts()
print(f"1. Data latih sampai {BATAS - pd.Timedelta(hours=1)}, data uji mulai {BATAS} (tidak dibuka), batas sama untuk 12 stasiun.")
print(f"2. Stasiun latih ({len(STASIUN_LATIH)}): {', '.join(STASIUN_LATIH)}.")
print(f"   Stasiun uji ({len(STASIUN_UJI)}, ditetapkan penulis, sama dengan aturan urutan jarak distribusi {URUTAN_UJI}): {', '.join(STASIUN_UJI)}.")
print(f"3. PM2.5 data latih (9 stasiun): median {pm.median():.0f}, rata-rata {pm.mean():.0f}, miring ke kanan "
      f"(skewness {pm.skew():.2f}). Galat besar saat episode tinggi akan mendominasi RMSE.")
print(f"4. Nilai hilang PM2.5 {latih['PM2.5'].isna().mean():.1%}; {(rentang_hilang <= 3).mean():.0%} rentang nilai hilang "
      f"<= 3 jam, terpanjang {rentang_hilang.max()} jam. Tidak diisi (FFILL_MAKS = {FFILL_MAKS}), diurus imputer di pipeline.")
print(f"5. Pola: tertinggi bulan {nama_bulan[per_bulan.idxmax()-1]}, jam {per_jam.idxmax()}:00.")
print(f"6. Korelasi PM2.5 jam t dengan jam t+24: {korelasi['PM2.5']:.2f}. "
      f"Fitur terkuat lain: {korelasi.drop('PM2.5').abs().idxmax()} ({korelasi.drop('PM2.5')[korelasi.drop('PM2.5').abs().idxmax()]:.2f}).")
print(f"7. {len(KOLOM_FITUR)} fitur siap. Baris latih {len(fitur_latih):,}; baris uji {len(fitur_uji):,} "
      f"({n_uji['stasiun_latih']:,} stasiun latih, {n_uji['stasiun_uji']:,} stasiun uji).")
print(f"8. Baseline naive forecast (CV 5 fold, 9 stasiun, RMSE rata-rata ± sd): "
      f"{tabel_baseline.loc['Naive forecast', 'RMSE rata-rata']:.1f} ± {tabel_baseline.loc['Naive forecast', 'RMSE sd']:.1f}.")
print(f"9. Model lanjutan (CV 5 fold, RMSE rata-rata ± sd): " +
      "; ".join(f"{n} {tabel5.loc[n, 'RMSE rata-rata']:.1f} ± {tabel5.loc[n, 'RMSE sd']:.1f}" for n in KANDIDAT) + ".")
print(f"   Kandidat menurut aturan protokol butir 2: {pilih} (belum final, data uji belum dibuka).")
print(f"10. Eksperimen lanjutan (05.5): RMSE validasi terendah {ringkas.iloc[0]['RMSE rata-rata']:.2f} ({ringkas.index[0]}), "
      f"terbaik tanpa lintas stasiun {tanpa_lintas.iloc[0]['RMSE rata-rata']:.2f}. Di bawah 70: {bool(ringkas.iloc[0]['RMSE rata-rata'] < 70)}. Data uji belum dibuka.")
print(f"11. Tuning model linear (05.7, 20 konfigurasi): kandidat menurut aturan {pilih7}, RMSE {ringkas7.loc[pilih7, 'RMSE rata-rata']:.2f} ± {ringkas7.loc[pilih7, 'RMSE sd']:.2f}, "
      f"MAE {ringkas7.loc[pilih7, 'MAE rata-rata']:.2f} ± {ringkas7.loc[pilih7, 'MAE sd']:.2f}. Data uji belum dibuka.")

**Yang perlu diputuskan atau dicek berikutnya**
* Tiga stasiun uji (Dingling, Wanliu, Shunyi) sudah ditetapkan penulis sebelum data uji dibuka. Mengubahnya berarti mengubah keputusan, dan *notebook* dijalankan ulang dari awal.
* Desain 9 stasiun latih dan 3 stasiun uji belum dikonfirmasi dosen.
* Bagian 05 sudah membandingkan model lanjutan dengan *baseline* pada *fold* yang sama (hanya *cross-validation*). Yang belum: *tuning* sistematis (terutama *gradient boosting*), persetujuan penulis atas aturan pemilihan di protokol, dan pembekuan konfigurasi model final sebelum data uji dibuka.
* Nilai ekstrem dipertahankan (keputusan 14 Nov 2026). Hasil 02.5 dipakai untuk mengecek apakah keputusan itu masih wajar.
* Berkas uji (`fitur_uji.csv`) dibuka satu kali di tahap 6 lewat `buka_data_uji`, dan skor dilaporkan terpisah untuk 9 stasiun latih dan 3 stasiun uji.